# Nebula checkpoint comparison and D2 Python foundation training

Runs the same 26 small tasks on the deployed 100M model's full-precision source, the later ~100M V28 subword foundation model, and the ~250M V30 foundation checkpoint at 7,000 updates. This is a task sample, not an intelligence benchmark. Coding checks inspect syntax and structure; generated code is never executed. No model is automatically published.

Then prepares a NEW PRIVATE 1B D2 Python foundation experiment from pinned CPython, Django, pytest and SymPy sources, retaining their full notices and source paths. Code is syntax checked, deduplicated and split into training/validation; private chats and old unknown-provenance data are excluded. Local preflight: 5,107 training code documents and 349 validation code documents, about 76 MB of unique text. This is a small foundation pilot, not enough to promise a useful 1B coding assistant.

The two-T4 experimental FSDP path runs two updates, uploads and verifies a checkpoint, then resumes for the remaining six-hour budget (15-minute save reserve). Stop duplicate trainers. Uses the existing NEBULA_HF_TOKEN secret. Models, corpus and checkpoints remain in RAM and private Hugging Face; code/packages use temporary disk. GPU, host RAM and private storage limits can still stop the run. Do not treat an unconfirmed upload as a saved checkpoint. No changes to the public website or existing trained models.


In [ ]:
import sys,subprocess,tempfile,os,json
from pathlib import Path
packages=Path(tempfile.gettempdir())/'nebula-vd2-packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(packages),'huggingface_hub>=0.30,<2','tokenizers==0.22.2','requests>=2.32,<3','numpy>=1.26,<3'],check=True)
sys.path.insert(0,str(packages))
# Use the GPU provider's supported PyTorch image. Torch is only needed for train/serve.
TOKEN=os.environ.get('HF_TOKEN','')
if not TOKEN:
    try:
        from google.colab import userdata
        TOKEN=userdata.get('HF_TOKEN')
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            TOKEN=UserSecretsClient().get_secret('NEBULA_HF_TOKEN')
        except Exception:
            pass
# Never print the token. Public inventory can run without it.


In [ ]:
SOURCES={'vd2_data.py': '"""V-D2 data admission, provenance, deduplication and holdout isolation. No torch dependency."""\nimport hashlib,json,re,io,gzip,struct,unicodedata\nSECRET=re.compile(r\'\\b(?:hf_|gsk_|sk-|AIza)[A-Za-z0-9_-]{18,}|-----BEGIN [A-Z ]*PRIVATE KEY-----|\\bBearer\\s+[A-Za-z0-9._-]{20,}\')\nSPECIAL=[\'<|n27_pad|>\',\'<|n27_bos|>\',\'<|n27_eos|>\',\'<|n27_system|>\',\'<|n27_user|>\',\'<|n27_assistant|>\',\'<|fim_prefix|>\',\'<|fim_suffix|>\',\'<|fim_middle|>\']\ndef sha(b):return hashlib.sha256(b).hexdigest()\n\ndef source_split(source):\n    # Historical manifests use camelCase names, including publicHoldout.\n    names=\' \'.join([str(source.get(\'key\',\'\')),str(source.get(\'state\',\'\')),str(source.get(\'entry\',{}).get(\'path\',\'\'))]+[str(p.get(\'path\',\'\')) for p in source.get(\'entry\',{}).get(\'parts\',[])]).lower()\n    return \'validation\' if source.get(\'split\')==\'validation\' or any(x in names for x in [\'validation\',\'heldout\',\'held-out\',\'held_out\',\'holdout\',\'hold-out\',\'hold_out\']) else \'train\'\ndef parameter_count(vocab=8192,width=1536,layers=35,context=2048):return vocab*width+context*width+layers*(12*width*width+13*width)+2*width\n\ndef pointers(value,trail=()):\n    if isinstance(value,dict):\n        if type(value.get(\'bytes\')) is int and isinstance(value.get(\'sha256\'),str) and (\'path\' in value or \'parts\' in value):yield trail,value\n        else:\n            for k,v in value.items():yield from pointers(v,trail+(str(k),))\n    elif isinstance(value,list):\n        for i,v in enumerate(value):yield from pointers(v,trail+(str(i),))\n\ndef candidates(repo,revision,state_path,state):\n    out=[]\n    for keys,entry in pointers(state):\n        name=\'/\'.join(keys);path=entry.get(\'path\') or entry.get(\'parts\',[{}])[0].get(\'path\',\'\');s=(name+\' \'+path).lower()\n        if any(x in s for x in [\'eval\',\'probe\',\'checks\',\'judge\',\'rules\',\'report\',\'card-\',\'license\',\'index\',\'sourcecard\',\'baseline\']):continue\n        kind=\'jsonl\' if \'.jsonl\' in path else \'bpe-u16\' if name in [\'trainTokens\',\'validationTokens\'] else \'bytes-u16\' if name.startswith(\'corpus/train/\') or name.startswith(\'corpus/validation/\') else None\n        if not kind:continue\n        source=dict(repo=repo,revision=revision,state=state_path,key=name,entry=entry,format=kind,tokenizer=state.get(\'tokenizer\') if kind==\'bpe-u16\' else None,approved=False,license=\'\',origin=\'historical-unreviewed\')\n        source[\'split\']=source_split(source);out.append(source)\n    return out\n\ndef texts(blob,kind,tokenizer=None):\n    if kind==\'jsonl\':\n        for line in blob.decode(\'utf-8\',errors=\'strict\').splitlines():\n            if not line.strip():continue\n            r=json.loads(line)\n            if not isinstance(r,dict):continue\n            if (isinstance(r.get(\'source\'),dict) and r[\'source\'].get(\'kind\')==\'user\') or r.get(\'origin\') in [\'user-chat\',\'review-feed\']:continue\n            if isinstance(r.get(\'text\'),str):yield r[\'text\']\n            elif isinstance(r.get(\'code\'),str):yield r[\'code\']\n            elif isinstance(r.get(\'messages\'),list):\n                # Preserve whole approved conversations as code documentation, not a new chat objective.\n                messages=r[\'messages\']\n                if all(isinstance(m,dict) and m.get(\'role\') in [\'system\',\'user\',\'assistant\'] and isinstance(m.get(\'content\'),str) for m in messages):\n                    yield \'\\n\\n\'.join(m[\'content\'] for m in messages)\n    elif kind in [\'bytes-u16\',\'bpe-u16\']:\n        if len(blob)%2:raise ValueError(\'Unaligned uint16 corpus\')\n        chunk=[]\n        for (token,) in struct.iter_unpack(\'<H\',blob):\n            boundary=token>=256 if kind==\'bytes-u16\' else token<6\n            if boundary:\n                if chunk:yield bytes(chunk).decode(\'utf-8\',errors=\'strict\') if kind==\'bytes-u16\' else tokenizer.decode(chunk,skip_special_tokens=True)\n                chunk=[]\n            else:chunk.append(token)\n        if chunk:yield bytes(chunk).decode(\'utf-8\',errors=\'strict\') if kind==\'bytes-u16\' else tokenizer.decode(chunk,skip_special_tokens=True)\n    else:raise ValueError(\'Unsupported encoding: \'+kind)\n\ndef clean(text):\n    text=unicodedata.normalize(\'NFC\',text.replace(\'\\r\\n\',\'\\n\')).strip()\n    if not 60<=len(text)<=150000 or SECRET.search(text) or \'\\ufffd\' in text or any(x in text for x in SPECIAL):return None\n    if any(ord(c)<32 and c not in \'\\n\\t\\r\' for c in text):return None\n    return text\n\ndef category(text):\n    if re.search(r\'(?m)^\\s*(?:def |class |import |from \\w+ import |function |export |const |let |public |package |#include|SELECT |CREATE TABLE)|```(?:python|py|javascript|js|typescript|ts|sql|rust|go|java|cpp|html)\',text):return \'code\'\n    if re.search(r\'\\b(?:unit test|regression test|SQL injection|authorization|authentication|parameterized|compiler|algorithm|data structure|HTTP|database|API|function|programming)\\b\',text,re.I):return \'docs\'\n    return \'general\'\n\nclass Corpus:\n    def __init__(self,max_bytes=256_000_000):self.records={};self.bytes=0;self.max_bytes=max_bytes;self.rejected={}\n    def add(self,text,source):\n        text=clean(text)\n        if text is None:self.rejected[\'invalid-or-secret\']=self.rejected.get(\'invalid-or-secret\',0)+1;return\n        if not source.get(\'approved\') or not source.get(\'license\') or source.get(\'origin\') in [\'user-chat\',\'review-feed\',\'historical-unreviewed\']:raise ValueError(\'Explicit provenance and license review required\')\n        fingerprint=sha(re.sub(r\'\\s+\',\' \',text).encode());split=\'validation\' if source_split(source)==\'validation\' or int(fingerprint[:8],16)%50==0 else \'train\'\n        if fingerprint in self.records:\n            if split==\'validation\':self.records[fingerprint][\'split\']=\'validation\'\n            return\n        size=len(text.encode())\n        if self.bytes+size>self.max_bytes:raise ValueError(\'RAM corpus budget reached; reduce source selection or increase deliberate budget\')\n        self.records[fingerprint]=dict(id=fingerprint,text=text,split=split,category=category(text),source={k:source[k] for k in [\'repo\',\'revision\',\'key\',\'license\',\'origin\']});self.bytes+=size\n    def summary(self):\n        counts={}\n        for r in self.records.values():k=r[\'split\']+\'/\'+r[\'category\'];counts[k]=counts.get(k,0)+1\n        return dict(documents=len(self.records),uniqueUtf8Bytes=self.bytes,counts=counts,rejected=self.rejected)\n    def payload(self):return \'\\n\'.join(json.dumps(r,ensure_ascii=False) for r in self.records.values()).encode()\n', 'vd2_hub.py': '"""Private, atomic HF artifacts: RAM buffers -> immutable parts -> manifest. No local model/data cache."""\nimport copy,hashlib,json,re,io\nfrom urllib.parse import quote\n\ndef sha(b):return hashlib.sha256(b).hexdigest()\nclass Hub:\n    def __init__(self,repo,token,run=\'nebula-vd2-1b-code\'):\n        import requests\n        from huggingface_hub import HfApi\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-vd2-[\\w-]{1,50}\',run):raise ValueError(\'Invalid output namespace\')\n        self.repo=repo;self.token=token;self.api=HfApi(token=token);self.session=requests.Session();self.prefix=\'experiments/\'+run+\'/\'\n        if self.api.whoami()[\'name\']!=repo.split(\'/\')[0]:raise ValueError(\'Output must belong to the token owner\')\n        self.api.create_repo(repo,repo_type=\'model\',private=True,exist_ok=True)\n        info=self.api.model_info(repo)\n        if not info.private:raise ValueError(\'Owner-only weights require a PRIVATE output repository; source repositories remain unchanged\')\n        self.revision=info.sha\n    def fetch(self,repo,path,revision,max_bytes=160000000):\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or \'..\' in path.split(\'/\') or path.startswith(\'/\'):raise ValueError(\'Invalid source path\')\n        url=f\'https://huggingface.co/{repo}/resolve/{quote(revision,safe="")}/{quote(path,safe="/")}\'\n        with self.session.get(url,headers={\'Authorization\':\'Bearer \'+self.token},timeout=(30,180),stream=True) as r:\n            r.raise_for_status();out=bytearray()\n            for chunk in r.iter_content(1024*1024):\n                out.extend(chunk)\n                if len(out)>max_bytes:raise ValueError(\'Object exceeds declared RAM limit\')\n            return bytes(out)\n    def read_entry(self,repo,revision,entry):\n        if not 0<entry[\'bytes\']<=20_000_000_000:raise ValueError(\'Invalid object size\')\n        out=io.BytesIO()\n        for p in entry.get(\'parts\',[entry]):\n            data=self.fetch(repo,p[\'path\'],revision,max_bytes=p[\'bytes\'])\n            if len(data)!=p[\'bytes\'] or sha(data)!=p[\'sha256\']:raise ValueError(\'Source checksum mismatch\')\n            out.write(data)\n        data=out.getvalue()\n        if len(data)!=entry[\'bytes\'] or sha(data)!=entry[\'sha256\']:raise ValueError(\'Combined checksum mismatch\')\n        return data\n    def state(self):\n        import requests\n        try:return json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision,2_000_000))\n        except requests.HTTPError as e:\n            if e.response.status_code==404:return None\n            raise\n    def publish(self,state,artifacts):\n        from huggingface_hub import CommitOperationAdd\n        next=copy.deepcopy(state);ops=[]\n        for key,value in artifacts.items():\n            view=value.getbuffer() if isinstance(value,io.BytesIO) else memoryview(value)\n            digest=hashlib.sha256(view).hexdigest();parts=[]\n            for i,start in enumerate(range(0,len(view),64*1024*1024)):\n                b=bytes(view[start:start+64*1024*1024]);path=self.prefix+\'objects/\'+digest+f\'/part-{i:04d}.bin\'\n                parts.append(dict(path=path,bytes=len(b),sha256=sha(b)));ops.append(CommitOperationAdd(path_in_repo=path,path_or_fileobj=b))\n            next[key]=dict(bytes=len(view),sha256=digest,parts=parts)\n        ops.append(CommitOperationAdd(path_in_repo=self.prefix+\'state.json\',path_or_fileobj=json.dumps(next,indent=2).encode()))\n        commit=self.api.create_commit(repo_id=self.repo,parent_commit=self.revision,operations=ops,commit_message=\'V-D2 owner-only data/checkpoint backup\')\n        self.revision=commit.oid\n        if json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision,2_000_000))!=next:raise ValueError(\'Backup verification failed\')\n        print(json.dumps(dict(vd2BackupConfirmed=True,revision=self.revision,updates=next.get(\'updates\',0),private=True,websiteUpdated=False)),flush=True)\n        return next\n', 'vd2_lab.py': '"""Coding-focused consolidation and conservative 1B training. Artifacts persist only in private HF."""\nimport copy,gzip,hashlib,io,json,math,os,random,time\nfrom pathlib import Path\nfrom vd2_data import candidates,Corpus,texts,sha,SPECIAL,parameter_count,source_split\nfrom vd2_hub import Hub\nMODEL=dict(vocab=8192,width=1536,heads=24,layers=35,context=2048)\nDEFAULT=dict(model=MODEL,updates=10000,batch=8,microbatch=1,learningRate=0.0001,warmup=200,saveEvery=100,evaluateEvery=500,seed=31025,maxCorpusBytes=256_000_000,minCodeDocuments=5000)\n\ndef engine_hash():\n    root=Path(__file__).parent\n    names=[\'vd2_data.py\',\'vd2_hub.py\',\'vd2_lab.py\',\'vd2_fsdp.py\',\'v30_expanded_model.py\',\'v30_device_audit.py\']\n    return sha(\'\'.join((root/n).read_text(encoding=\'utf-8\') for n in names).encode())\n\ndef inventory(repos,token=None):\n    from huggingface_hub import HfApi\n    import requests\n    api=HfApi(token=token);entries=[];runs=[]\n    for repo in repos:\n        info=api.model_info(repo);revision=info.sha\n        for name in api.list_repo_files(repo,revision=revision):\n            if not name.startswith(\'experiments/\') or not name.endswith(\'/state.json\'):continue\n            url=f\'https://huggingface.co/{repo}/resolve/{revision}/{name}\'\n            r=requests.get(url,headers={\'Authorization\':\'Bearer \'+token} if token else {},timeout=60);r.raise_for_status()\n            if len(r.content)>2_000_000:raise ValueError(\'Manifest too large\')\n            state=r.json();runs.append(dict(repo=repo,revision=revision,path=name));entries+=candidates(repo,revision,name,state)\n    # Content duplicates share a candidate; source references remain recorded in the inventory.\n    return dict(format=\'nebula-vd2-inventory-1\',runs=runs,candidates=entries,notScanned=[\'Unconfigured GCS/S3 buckets\',\'Private repositories not accessible to supplied token\'],noObjectsMoved=True)\n\ndef prepare(repo,token,approved,config=DEFAULT):\n    from tokenizers import Tokenizer,models,pre_tokenizers,decoders,trainers\n    hub=Hub(repo,token);existing=hub.state()\n    fingerprint=sha(json.dumps({\'config\':config,\'sources\':approved,\'engine\':engine_hash()},sort_keys=True).encode())\n    if existing:\n        if existing.get(\'signature\')!=fingerprint:raise ValueError(\'Changed source/config: choose a new output repository/run, do not overwrite prior work\')\n        if existing.get(\'dataReady\'):return existing\n    if not approved or any(not x.get(\'approved\') or not x.get(\'license\') or x.get(\'origin\') in [\'historical-unreviewed\',\'user-chat\',\'review-feed\'] for x in approved):raise ValueError(\'Review and approve explicit training provenance/licenses first; do not admit private chats\')\n    corpus=Corpus(config[\'maxCorpusBytes\'])\n    # Process historical validation first. Dedup moves any duplicated training record to validation.\n    for source in sorted(approved,key=lambda x:source_split(x)!=\'validation\'):\n        blob=hub.read_entry(source[\'repo\'],source[\'revision\'],source[\'entry\']);tokenizer=None\n        if source[\'format\']==\'bpe-u16\':\n            if not source.get(\'tokenizer\'):raise ValueError(\'Cannot decode subwords without their exact tokenizer\')\n            tokenizer=Tokenizer.from_str(hub.read_entry(source[\'repo\'],source[\'revision\'],source[\'tokenizer\']).decode())\n        for text in texts(blob,source[\'format\'],tokenizer):corpus.add(text,source)\n        print(json.dumps({\'vd2Consolidating\':source[\'key\'],**corpus.summary()}),flush=True)\n        del blob\n    if corpus.summary()[\'counts\'].get(\'train/code\',0)<config[\'minCodeDocuments\']:raise ValueError(\'Not enough distinct code documents for this coding experiment. Historical story/chat data is not an adequate coding corpus.\')\n    tokenizer=Tokenizer(models.BPE(unk_token=None));tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False);tokenizer.decoder=decoders.ByteLevel()\n    tokenizer.train_from_iterator((r[\'text\'] for r in corpus.records.values() if r[\'split\']==\'train\' and r[\'category\'] in [\'code\',\'docs\']),trainers.BpeTrainer(vocab_size=config[\'model\'][\'vocab\'],special_tokens=SPECIAL,initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))\n    if tokenizer.get_vocab_size()!=config[\'model\'][\'vocab\']:raise ValueError(\'Corpus too small to build the target vocabulary\')\n    state=dict(format=\'nebula-vd2-code-1\',signature=fingerprint,engine=engine_hash(),config=config,dataReady=True,updates=0,qualityApproved=False,websiteUpdated=False,summary=corpus.summary(),sources=approved)\n    return hub.publish(state,{\'corpus\':gzip.compress(corpus.payload()),\'tokenizer\':tokenizer.to_str().encode()})\n\ndef train(repo,token,hours=6.0,reserve_minutes=15,config=DEFAULT):\n    import numpy as np\n    import torch\n    import torch.nn.functional as F\n    from tokenizers import Tokenizer\n    from v30_expanded_model import Model,Config\n    from v30_device_audit import audit\n    # Full-parameter AdamW needs much more than 16GB. Two visible T4s do not pool memory automatically.\n    if not torch.cuda.is_available():raise ValueError(\'No CUDA GPU. Preparation/inventory can run on CPU; 1B training is not started.\')\n    props=torch.cuda.get_device_properties(0)\n    if props.total_memory<38*1024**3:raise ValueError(\'This implementation requires one >=40GB GPU for full 1B AdamW training. 2xT4 is not one pooled GPU. Use a larger GPU or implement/test sharded training separately; no purchase is made.\')\n    if hours<=reserve_minutes/60:raise ValueError(\'Reserve exceeds session duration\')\n    hub=Hub(repo,token);state=hub.state()\n    if not state or not state.get(\'dataReady\') or state[\'config\']!=config or state.get(\'engine\')!=engine_hash():raise ValueError(\'Prepare and freeze the approved coding corpus/config first\')\n    deadline=time.monotonic()+hours*3600-reserve_minutes*60\n    tokenizer_blob=hub.read_entry(repo,hub.revision,state[\'tokenizer\']);tokenizer=Tokenizer.from_str(tokenizer_blob.decode())\n    records=[json.loads(line) for line in gzip.decompress(hub.read_entry(repo,hub.revision,state[\'corpus\'])).decode().splitlines()]\n    pools={split:{kind:[] for kind in [\'code\',\'docs\',\'general\']} for split in [\'train\',\'validation\']}\n    for r in records:\n        ids=tokenizer.encode(r[\'text\'],add_special_tokens=False).ids+[2]\n        pools[r[\'split\']][r[\'category\']].append(ids)\n    if not pools[\'validation\'][\'code\']:raise ValueError(\'Independent coding validation documents required\')\n    torch.manual_seed(config[\'seed\']);random.seed(config[\'seed\']);model=Model(Config(**config[\'model\'])).cuda()\n    if sum(p.numel() for p in model.parameters())!=parameter_count(**{k:v for k,v in config[\'model\'].items() if k!=\'heads\'}):raise ValueError(\'Parameter count mismatch\')\n    bf16=torch.cuda.is_bf16_supported();dtype=torch.bfloat16 if bf16 else torch.float16\n    optimizer=torch.optim.AdamW([{\'params\':[p for p in model.parameters() if p.ndim>=2],\'weight_decay\':.1},{\'params\':[p for p in model.parameters() if p.ndim<2],\'weight_decay\':0}],lr=config[\'learningRate\'],betas=(.9,.95))\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=not bf16);step=state[\'updates\']\n    if state.get(\'working\'):\n        saved=torch.load(io.BytesIO(hub.read_entry(repo,hub.revision,state[\'working\'])),map_location=\'cpu\',weights_only=True)\n        if saved[\'signature\']!=state[\'signature\']:raise ValueError(\'Checkpoint configuration mismatch\')\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\']);random.setstate(saved[\'random\']);torch.set_rng_state(saved[\'torchRng\']);torch.cuda.set_rng_state(saved[\'cudaRng\']);del saved\n    # Same causal implementation as V30; evaluate/train agreement is checked before updates.\n    check_tokens=np.array((pools[\'train\'][\'code\'][0]*4)[:256],dtype=np.uint16)\n    report=audit(model,check_tokens)\n    if not report[\'passed\']:\n        model.set_attention_backend(\'math\');report=audit(model,check_tokens)\n        if not report[\'passed\']:raise ValueError(\'Device audit failed. No optimizer updates started.\')\n    state[\'deviceAudit\']=report\n    def example(split,seed):\n        rng=random.Random(seed);available=pools[split];kind=rng.choices([\'code\',\'docs\',\'general\'],[.75 if available[\'code\'] else 0,.2 if available[\'docs\'] else 0,.05 if available[\'general\'] else 0])[0]\n        data=[]\n        while len(data)<config[\'model\'][\'context\']+1:\n            ids=list(rng.choice(available[kind]))\n            if kind==\'code\' and len(ids)>64 and rng.random()<.2:\n                a,b=sorted(rng.sample(range(1,len(ids)-1),2));ids=[6]+ids[:a]+[7]+ids[b:]+[8]+ids[a:b]+[2]\n            data.extend(ids)\n        return torch.tensor(data[:config[\'model\'][\'context\']+1],device=\'cuda\',dtype=torch.long)\n    @torch.inference_mode()\n    def evaluate():\n        model.eval();losses=[]\n        for i in range(16):\n            ids=example(\'validation\',config[\'seed\']+900000+i)\n            with torch.autocast(\'cuda\',dtype=dtype):losses.append(float(F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])))\n        return dict(loss=sum(losses)/len(losses),targets=16*config[\'model\'][\'context\'],qualityApproved=False,notice=\'Held-out coding loss is not proof of correct or secure code. Run separate sandboxed tests before serving.\')\n    def save():\n        nonlocal state\n        buffer=io.BytesIO();torch.save(dict(format=\'nebula-vd2-code-1\',signature=state[\'signature\'],model={k:v.detach().cpu() for k,v in model.state_dict().items()},optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),random=random.getstate(),torchRng=torch.get_rng_state(),cudaRng=torch.cuda.get_rng_state()),buffer)\n        state=hub.publish({**state,\'updates\':step,\'supervisedTokens\':step*config[\'batch\']*config[\'model\'][\'context\']},{\'working\':buffer});buffer.close()\n    last=time.monotonic();last_saved=step\n    if not state.get(\'baseline\'):state=hub.publish({**state,\'baseline\':evaluate()}, {})\n    try:\n        while step<config[\'updates\'] and time.monotonic()<deadline:\n            model.train();optimizer.zero_grad(set_to_none=True);lr=config[\'learningRate\']*min(1,(step+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*step/config[\'updates\'])))\n            for g in optimizer.param_groups:g[\'lr\']=lr\n            losses=[]\n            for micro in range(config[\'batch\']):\n                ids=example(\'train\',config[\'seed\']+step*config[\'batch\']+micro)\n                with torch.autocast(\'cuda\',dtype=dtype):loss=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])/config[\'batch\']\n                if not torch.isfinite(loss):raise ValueError(\'Nonfinite loss; no successful checkpoint is replaced\')\n                scaler.scale(loss).backward();losses.append(float(loss.detach()))\n            scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1,error_if_nonfinite=True);scaler.step(optimizer);scaler.update();step+=1\n            if step%25==0:print(json.dumps(dict(vd2Update=step,parameters=parameter_count(),loss=sum(losses),lr=lr)),flush=True)\n            if step%config[\'evaluateEvery\']==0:state[\'evaluation\']=evaluate()\n            if step%config[\'saveEvery\']==0 or time.monotonic()-last>=180:\n                save();last_saved=step;last=time.monotonic()\n    except KeyboardInterrupt:print(\'Stop requested; saving this exact trajectory.\',flush=True)\n    # Upload errors propagate and stop the run; do not silently keep training without backups.\n    if last_saved!=step:save()\n    state=hub.publish({**state,\'evaluation\':evaluate()}, {})\n    return dict(updatesSaved=step,parameters=parameter_count(),complete=step>=config[\'updates\'],qualityApproved=False,websiteUpdated=False)\n', 'vd2_fsdp.py': '"""EXPERIMENTAL two-GPU FULL_SHARD trainer. Requires a real 2xT4 memory/resume smoke test.\nEach rank\'s optimizer is saved separately; only rank 0 atomically advances the model manifest.\nArtifacts remain in RAM -> private HF. Do not run two torchrun jobs for one output run.\n"""\nimport argparse,gc,gzip,io,json,os,random,time\nfrom datetime import timedelta\nfrom functools import partial\nfrom vd2_hub import Hub\nfrom vd2_lab import DEFAULT,engine_hash\n\ndef run(repo,token,hours=6,reserve_minutes=15,max_session_updates=0):\n    import numpy as np\n    import torch\n    import torch.distributed as dist\n    import torch.nn.functional as F\n    from tokenizers import Tokenizer\n    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP,ShardingStrategy,MixedPrecision,StateDictType,FullStateDictConfig\n    from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy\n    from torch.distributed.fsdp.sharded_grad_scaler import ShardedGradScaler\n    from v30_expanded_model import Model,Config,Block\n    from v30_device_audit import audit\n    rank=int(os.environ.get(\'RANK\',-1));local=int(os.environ.get(\'LOCAL_RANK\',-1));world=int(os.environ.get(\'WORLD_SIZE\',0))\n    if world!=2 or not torch.cuda.is_available() or torch.cuda.device_count()<2:raise ValueError(\'Launch with torchrun --standalone --nproc_per_node=2 vd2_fsdp.py; requires two actual CUDA GPUs\')\n    torch.cuda.set_device(local);device=torch.device(\'cuda\',local);dist.init_process_group(\'nccl\',device_id=device,timeout=timedelta(minutes=30))\n    try:\n        hub=Hub(repo,token);state=hub.state()\n        if not state or not state.get(\'dataReady\') or state.get(\'engine\')!=engine_hash():raise ValueError(\'Prepare the frozen approved corpus first\')\n        config=state[\'config\'];step=state.get(\'updates\',0);session_target=min(config[\'updates\'],step+max_session_updates) if max_session_updates else config[\'updates\']\n        if state.get(\'worldSize\',2)!=2:raise ValueError(\'Cannot resume a single-GPU optimizer as two FSDP optimizers; use a separate deliberate weight-transfer experiment\')\n        if config[\'batch\']%2:raise ValueError(\'Global batch must divide by two ranks\')\n        if not 0<reserve_minutes/60<hours:raise ValueError(\'Invalid session/reserve\')\n        deadline=time.monotonic()+hours*3600-reserve_minutes*60\n        tokenizer=Tokenizer.from_str(hub.read_entry(repo,hub.revision,state[\'tokenizer\']).decode())\n        pools={s:{k:[] for k in [\'code\',\'docs\',\'general\']} for s in [\'train\',\'validation\']}\n        records=gzip.decompress(hub.read_entry(repo,hub.revision,state[\'corpus\'])).decode()\n        for line in records.splitlines():\n            r=json.loads(line);pools[r[\'split\']][r[\'category\']].append(tokenizer.encode(r[\'text\'],add_special_tokens=False).ids+[2])\n        del records;gc.collect()\n        if not pools[\'train\'][\'code\'] or not pools[\'validation\'][\'code\']:raise ValueError(\'Independent coding training and validation documents are required\')\n        torch.manual_seed(config[\'seed\']);base=Model(Config(**config[\'model\']))\n        if state.get(\'working\'):\n            blob=hub.read_entry(repo,hub.revision,state[\'working\']);saved=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n            if saved[\'signature\']!=state[\'signature\']:raise ValueError(\'Model signature mismatch\')\n            base.load_state_dict(saved[\'model\']);del blob,saved;gc.collect()\n        # Before optimizer allocation, audit the actual full model/device (small sequence).\n        base.to(device);probe=np.array((pools[\'train\'][\'code\'][0]*4)[:128],dtype=np.uint16);report=audit(base,probe)\n        if not report[\'passed\']:\n            base.set_attention_backend(\'math\');report=audit(base,probe)\n            if not report[\'passed\']:raise ValueError(\'Device audit failed before FSDP updates\')\n        if rank==0:print(json.dumps(dict(vd2FSDPAudit=report,experimental=True)),flush=True)\n        model=FSDP(base,auto_wrap_policy=partial(transformer_auto_wrap_policy,transformer_layer_cls={Block}),sharding_strategy=ShardingStrategy.FULL_SHARD,mixed_precision=MixedPrecision(param_dtype=torch.float16,reduce_dtype=torch.float32,buffer_dtype=torch.float16),device_id=device,use_orig_params=True,limit_all_gathers=True)\n        optimizer=torch.optim.AdamW([{\'params\':[p for n,p in model.named_parameters() if not any(s in n for s in [\'bias\',\'norm\'])],\'weight_decay\':.1},{\'params\':[p for n,p in model.named_parameters() if any(s in n for s in [\'bias\',\'norm\'])],\'weight_decay\':0}],lr=config[\'learningRate\'],betas=(.9,.95),foreach=False)\n        scaler=ShardedGradScaler(enabled=True)\n        opt_key=\'optimizer\'+str(rank)\n        if step:\n            if not state.get(opt_key):raise ValueError(\'Missing rank optimizer; refuse an incomplete resume\')\n            saved=torch.load(io.BytesIO(hub.read_entry(repo,hub.revision,state[opt_key])),map_location=\'cpu\',weights_only=True)\n            if saved[\'step\']!=step or saved[\'worldSize\']!=2 or saved[\'torchVersion\']!=str(torch.__version__):raise ValueError(\'Rank optimizer/version mismatch\')\n            optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\']);random.setstate(saved[\'random\']);torch.set_rng_state(saved[\'torchRng\']);torch.cuda.set_rng_state(saved[\'cudaRng\'],device);del saved;gc.collect()\n        def example(split,seed):\n            rng=random.Random(seed);p=pools[split];kind=rng.choices([\'code\',\'docs\',\'general\'],[.75 if p[\'code\'] else 0,.2 if p[\'docs\'] else 0,.05 if p[\'general\'] else 0])[0];data=[]\n            while len(data)<config[\'model\'][\'context\']+1:\n                ids=list(rng.choice(p[kind]))\n                if split==\'train\' and kind==\'code\' and len(ids)>64 and rng.random()<.2:\n                    a,b=sorted(rng.sample(range(1,len(ids)-1),2));ids=[6]+ids[:a]+[7]+ids[b:]+[8]+ids[a:b]+[2]\n                data.extend(ids)\n            return torch.tensor(data[:config[\'model\'][\'context\']+1],device=device,dtype=torch.long)\n        @torch.inference_mode()\n        def evaluate():\n            model.eval();total=torch.zeros(1,device=device)\n            for i in range(4):\n                ids=example(\'validation\',config[\'seed\']+900000+rank*4+i)\n                with torch.autocast(\'cuda\',dtype=torch.float16):total+=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])\n            dist.all_reduce(total);return dict(loss=float(total)/(4*world),targets=8*config[\'model\'][\'context\'],qualityApproved=False)\n        def save():\n            nonlocal state\n            # Collect weights only: no rank ever gathers the full 8GB optimizer state.\n            with FSDP.state_dict_type(model,StateDictType.FULL_STATE_DICT,FullStateDictConfig(offload_to_cpu=True,rank0_only=True)):weights=model.state_dict()\n            opt=io.BytesIO();torch.save(dict(step=step,worldSize=2,torchVersion=str(torch.__version__),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),random=random.getstate(),torchRng=torch.get_rng_state(),cudaRng=torch.cuda.get_rng_state(device)),opt)\n            # Rank 1 uses a separate manifest. It cannot accidentally advance the main run\'s step.\n            metadata=[None]\n            if rank==1:\n                sibling=Hub(repo,token,run=\'nebula-vd2-1b-code-rank1\');s=sibling.publish({\'updates\':step,\'signature\':state[\'signature\']},{\'optimizer1\':opt});metadata[0]={\'revision\':sibling.revision,\'entry\':s[\'optimizer1\']}\n            dist.broadcast_object_list(metadata,src=1,device=device);opt1=metadata[0];opt.close() if rank==1 else None\n            if rank==0:\n                hub.revision=opt1[\'revision\'];model_blob=io.BytesIO();torch.save(dict(format=\'nebula-vd2-code-1\',signature=state[\'signature\'],model=weights),model_blob)\n                state=hub.publish({**state,\'updates\':step,\'worldSize\':2,\'optimizer1\':opt1[\'entry\'],\'supervisedTokens\':step*config[\'batch\']*config[\'model\'][\'context\'],\'experimentalFSDP\':True},{\'working\':model_blob,\'optimizer0\':opt});model_blob.close();opt.close()\n            del weights;gc.collect()\n            metadata=[{\'revision\':hub.revision,\'state\':state} if rank==0 else None];dist.broadcast_object_list(metadata,src=0,device=device);hub.revision=metadata[0][\'revision\'];state=metadata[0][\'state\'];dist.barrier(device_ids=[local])\n        if rank==0:print(json.dumps(dict(vd2FSDP=True,worldSize=2,precision=\'fp16\',fullShard=True,resumedUpdates=step,experimental=True)),flush=True)\n        state[\'evaluation\']=evaluate();last=time.monotonic();last_saved=step\n        while step<session_target:\n            keep=torch.tensor([time.monotonic()<deadline],device=device,dtype=torch.int32);dist.all_reduce(keep,op=dist.ReduceOp.MIN)\n            if not int(keep):break\n            model.train();optimizer.zero_grad(set_to_none=True)\n            import math\n            lr=config[\'learningRate\']*min(1,(step+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*step/config[\'updates\'])))\n            for group in optimizer.param_groups:group[\'lr\']=lr\n            loss_sum=0.\n            # Do not use no_sync(): it keeps full accumulated gradients and can defeat memory savings.\n            for micro in range(config[\'batch\']//world):\n                ids=example(\'train\',config[\'seed\']+step*config[\'batch\']+rank*(config[\'batch\']//world)+micro)\n                with torch.autocast(\'cuda\',dtype=torch.float16):loss=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])/(config[\'batch\']//world)\n                if not torch.isfinite(loss):raise ValueError(\'Nonfinite loss; previous confirmed snapshot remains intact\')\n                scaler.scale(loss).backward();loss_sum+=float(loss.detach())\n            scaler.unscale_(optimizer);model.clip_grad_norm_(1.);old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()<old_scale:\n                if rank==0:print(json.dumps(dict(vd2FSDPSkippedOverflow=True,scale=scaler.get_scale())),flush=True)\n                continue\n            step+=1\n            if rank==0 and step%25==0:print(json.dumps(dict(vd2FSDPUpdate=step,lossRank0=loss_sum,worldSize=2,peakRank0GpuBytes=torch.cuda.max_memory_allocated(device))),flush=True)\n            if step%config[\'evaluateEvery\']==0:state[\'evaluation\']=evaluate()\n            should_save=torch.tensor([step%config[\'saveEvery\']==0 or time.monotonic()-last>=180],device=device,dtype=torch.int32);dist.all_reduce(should_save,op=dist.ReduceOp.MAX)\n            if int(should_save):save();last_saved=step;last=time.monotonic()\n        if last_saved!=step:save()\n        state[\'evaluation\']=evaluate()\n        if rank==0:hub.publish(state,{})\n    finally:dist.destroy_process_group()\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--repo\',required=True);p.add_argument(\'--hours\',type=float,default=6);p.add_argument(\'--reserve-minutes\',type=float,default=15);p.add_argument(\'--max-session-updates\',type=int,default=0);args=p.parse_args();run(args.repo,os.environ[\'HF_TOKEN\'],args.hours,args.reserve_minutes,args.max_session_updates)\n', 'vd2_serve.py': '"""Private inference worker. No website crawling, execution or permission-grant tools here."""\nimport hmac,io,json,os,hashlib\n\ndef create_app():\n    import torch\n    from fastapi import FastAPI,Header,HTTPException,Request\n    from tokenizers import Tokenizer\n    from v30_expanded_model import Model,Config,encode_chat\n    from vd2_hub import Hub\n    # Request is looked up from globals by FastAPI on some releases.\n    globals()[\'Request\']=Request\n    token=os.environ.get(\'NEBULA_VD2_WORKER_TOKEN\',\'\')\n    if len(token)<32:raise RuntimeError(\'Configure a strong private worker token\')\n    hub=Hub(os.environ[\'NEBULA_VD2_REPO\'],os.environ[\'HF_TOKEN\']);state=hub.state()\n    if not state or not state.get(\'working\'):raise RuntimeError(\'No trained V-D2 weights exist\')\n    if not state.get(\'qualityApproved\') and os.environ.get(\'NEBULA_VD2_ALLOW_EXPERIMENTAL\')!=\'true\':raise RuntimeError(\'Unapproved checkpoint. Complete coding evaluation or explicitly enable owner-only experimental serving.\')\n    blob=hub.read_entry(hub.repo,hub.revision,state[\'working\']);saved=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    model=Model(Config(**state[\'config\'][\'model\']));model.load_state_dict(saved[\'model\']);del saved,blob\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\';model.to(device).eval();tokenizer=Tokenizer.from_str(hub.read_entry(hub.repo,hub.revision,state[\'tokenizer\']).decode());checkpoint=\'sha256:\'+state[\'working\'][\'sha256\']\n    import asyncio\n    lock=asyncio.Lock();app=FastAPI(docs_url=None,redoc_url=None,openapi_url=None)\n    @app.post(\'/generate\')\n    async def generate(request:Request,authorization:str=Header(default=\'\')):\n        if not hmac.compare_digest(authorization,\'Bearer \'+token):raise HTTPException(401,\'Unauthorized\')\n        body=bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body)>120000:raise HTTPException(413,\'Request too large\')\n        try:\n            value=json.loads(body);messages=value[\'messages\'];limit=value[\'maxNewTokens\']\n            if value[\'model\']!=\'nebula-v-d2\' or len(messages)!=2 or messages[0][\'role\']!=\'system\' or messages[1][\'role\']!=\'user\' or type(limit)is not int or not 1<=limit<=2048:raise ValueError()\n            ids,_=encode_chat(tokenizer,messages,answer_prefix=True)\n        except Exception:raise HTTPException(400,\'Invalid generation request\')\n        if len(ids)>=model.config.context-32:raise HTTPException(413,\'Prompt exceeds context; send a smaller code excerpt\')\n        if lock.locked():raise HTTPException(429,\'Worker busy\')\n        def complete():\n            out=[]\n            with torch.inference_mode():\n                for _ in range(min(limit,model.config.context-len(ids))):\n                    x=torch.tensor([ids+out],device=device)\n                    logits=model(x)[0,-1].float();logits[[0,1,3,4,5,6,7,8]]=-float(\'inf\');next=int(logits.argmax())\n                    if next==2:break\n                    out.append(next)\n            return tokenizer.decode(out,skip_special_tokens=True)\n        async with lock:text=await asyncio.to_thread(complete)\n        return dict(model=\'nebula-v-d2\',checkpoint=checkpoint,text=text,experimental=not state.get(\'qualityApproved\',False))\n    return app\n', 'v30_expanded_model.py': '"""Depth-expanded subword model with explicit causal attention and inherited weights."""\nimport io\nimport math\nfrom dataclasses import dataclass,asdict\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint as activation_checkpoint\n\nSPECIAL=[\'<|n27_pad|>\',\'<|n27_bos|>\',\'<|n27_eos|>\',\'<|n27_system|>\',\'<|n27_user|>\',\'<|n27_assistant|>\']\nPAD,BOS,EOS,SYSTEM,USER,ASSISTANT=range(6)\nROLES={\'system\':SYSTEM,\'user\':USER,\'assistant\':ASSISTANT}\n@dataclass\nclass Config:\n    vocab:int=8192\n    width:int=768\n    heads:int=12\n    layers:int=34\n    context:int=1024\n\nclass Block(nn.TransformerEncoderLayer):\n    # Keep legacy parameter names, but bypass eval-only fused encoder/MHA paths.\n    def forward(self,x,src_mask=None):\n        z=self.norm1(x);b,n,w=z.shape;h=self.self_attn.num_heads;d=w//h\n        q,k,v=F.linear(z,self.self_attn.in_proj_weight,self.self_attn.in_proj_bias).chunk(3,dim=-1)\n        q,k,v=[t.reshape(b,n,h,d).transpose(1,2) for t in (q,k,v)]\n        if getattr(self,\'attention_backend\',\'default\')==\'math\':\n            from torch.nn.attention import sdpa_kernel,SDPBackend\n            with sdpa_kernel(SDPBackend.MATH):\n                a=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=True)\n        else:a=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=True)\n        a=a.transpose(1,2).contiguous().reshape(b,n,w)\n        x=x+self.self_attn.out_proj(a)\n        return x+self.linear2(F.gelu(self.linear1(self.norm2(x))))\n\nclass Model(nn.Module):\n    def __init__(self,config):\n        super().__init__();self.config=config\n        self.token=nn.Embedding(config.vocab,config.width)\n        self.position=nn.Embedding(config.context,config.width)\n        self.blocks=nn.ModuleList(Block(config.width,config.heads,config.width*4,\n            dropout=0.,activation=\'gelu\',batch_first=True,norm_first=True) for _ in range(config.layers))\n        self.norm=nn.LayerNorm(config.width)\n        self.output=nn.Linear(config.width,config.vocab,bias=False);self.output.weight=self.token.weight\n        self.register_buffer(\'mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.gradient_checkpointing=True\n        self.apply(self.initialize)\n        for block in self.blocks:\n            nn.init.normal_(block.self_attn.out_proj.weight,std=.02/math.sqrt(2*config.layers))\n            nn.init.normal_(block.linear2.weight,std=.02/math.sqrt(2*config.layers))\n    def set_attention_backend(self,backend):\n        if backend not in [\'default\',\'math\']:raise ValueError(\'Unsupported attention backend\')\n        for block in self.blocks:block.attention_backend=backend\n\n    @staticmethod\n    def initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if getattr(module,\'bias\',None) is not None:nn.init.zeros_(module.bias)\n    def forward(self,tokens):\n        n=tokens.shape[1]\n        if n>self.config.context:raise ValueError(\'Context exceeded\')\n        x=self.token(tokens)+self.position(torch.arange(n,device=tokens.device))\n        for block in self.blocks:\n            if self.training and torch.is_grad_enabled() and self.gradient_checkpointing:\n                x=activation_checkpoint(block,x,use_reentrant=False)\n            else:x=block(x)\n        return self.output(self.norm(x))\n\n\ndef encode_chat(tokenizer,messages,answer_prefix=False):\n    tokens=[BOS];learn=[False]\n    for message in messages:\n        role=message[\'role\'];text=message[\'content\']\n        if role not in ROLES or any(s in text for s in SPECIAL):raise ValueError(\'Invalid chat role or reserved marker\')\n        ids=tokenizer.encode(text,add_special_tokens=False).ids\n        tokens+=[ROLES[role]]+ids+[EOS]\n        learn+=[False]+[role==\'assistant\']*(len(ids)+1)\n    if answer_prefix:tokens+=[ASSISTANT];learn+=[False]\n    return tokens,learn\n\n\ndef chat_window(tokenizer,messages,context):\n    tokens,learn=encode_chat(tokenizer,messages)\n    if len(tokens)>context+1:raise ValueError(\'Complete conversation does not fit\')\n    x=torch.tensor(tokens[:-1],dtype=torch.long)\n    y=torch.tensor([v if flag else -100 for v,flag in zip(tokens[1:],learn[1:])],dtype=torch.long)\n    if not (y!=-100).any():raise ValueError(\'No assistant targets\')\n    return x,y\n\n\ndef checkpoint(model,training,tokenizer_sha):\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Nonfinite checkpoint\')\n    stream=io.BytesIO()\n    torch.save({\'format\':\'nebula-v30-expanded-subword-1\',\'config\':asdict(model.config),\'model\':model.state_dict(),\n        \'tokenizerSha256\':tokenizer_sha,\'training\':training},stream)\n    return stream\n\n\ndef restore(blob,tokenizer_sha):\n    value=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if value.get(\'format\')!=\'nebula-v30-expanded-subword-1\' or value[\'tokenizerSha256\']!=tokenizer_sha:\n        raise ValueError(\'Incompatible model/tokenizer; use the V30 checkpoint format\')\n    model=Model(Config(**value[\'config\']));model.load_state_dict(value[\'model\'])\n    return model,value[\'training\']\n\n\n@torch.inference_mode()\ndef generate(model,tokenizer,prompt,chat=False,limit=96):\n    model.eval()\n    ids=encode_chat(tokenizer,[{\'role\':\'user\',\'content\':prompt}],True)[0] if chat else [BOS]+tokenizer.encode(prompt).ids\n    if len(ids)>=model.config.context:raise ValueError(\'Probe too long\')\n    device=next(model.parameters()).device;out=[]\n    for _ in range(min(limit,model.config.context-len(ids))):\n        x=torch.tensor([ids+out],device=device)\n        logits=model(x)[0,-1].float();logits[[PAD,BOS,SYSTEM,USER,ASSISTANT]]=-float(\'inf\')\n        token=int(logits.argmax())\n        if token==EOS:break\n        out.append(token)\n    return tokenizer.decode(out,skip_special_tokens=True)\n\n\ndef expand_parent(blob,tokenizer_sha,target):\n    value=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if value.get(\'format\')!=\'nebula-v27-subword-1\' or value[\'tokenizerSha256\']!=tokenizer_sha:\n        raise ValueError(\'Expected original V28 subword weights and exact tokenizer\')\n    original=Config(**value[\'config\'])\n    for key in [\'vocab\',\'width\',\'heads\',\'context\']:\n        if getattr(original,key)!=getattr(target,key):raise ValueError(\'Depth expansion cannot change \'+key)\n    if target.layers<=original.layers:raise ValueError(\'Expansion must add layers\')\n    model=Model(target);weights=model.state_dict()\n    for key,tensor in value[\'model\'].items():\n        if key not in weights or weights[key].shape!=tensor.shape:raise ValueError(\'Legacy parameter mismatch: \'+key)\n        weights[key]=tensor\n    model.load_state_dict(weights)\n    with torch.no_grad():\n        for block in model.blocks[original.layers:]:\n            block.self_attn.out_proj.weight.zero_();block.self_attn.out_proj.bias.zero_()\n            block.linear2.weight.zero_();block.linear2.bias.zero_()\n    return model,dict(originalLayers=original.layers,expandedLayers=target.layers,\n        inheritedWeights=True,newResidualOutputsInitiallyZero=True,parentOptimizerDiscarded=True,\n        parameters=sum(p.numel() for p in model.parameters()))\n', 'v30_device_audit.py': '"""Check actual device/model masking and train/eval consistency before updates."""\nimport numpy as np\nimport torch\n\n@torch.inference_mode()\ndef audit(model,values):\n    original_mode=model.training;tf32=torch.backends.cuda.matmul.allow_tf32\n    device=next(model.parameters()).device;checks={};errors={}\n    try:\n        torch.backends.cuda.matmul.allow_tf32=False\n        n=min(64,model.config.context,len(values)-1)\n        seq=torch.tensor(np.array(values[:n],dtype=np.int64),device=device)[None]\n        target=torch.tensor(np.array(values[1:n+1],dtype=np.int64),device=device)[None]\n        checks[\'one_token_shift\']=torch.equal(seq[:,1:],target[:,:-1])\n        model.eval();full=model(seq).float()\n        model.train()\n        with torch.inference_mode(False),torch.enable_grad():\n            training=model(seq.clone()).float().detach()\n        errors[\'trainEvalMaxAbs\']=float((full-training).abs().max())\n        checks[\'train_eval_agree\']=torch.allclose(full,training,atol=1e-3,rtol=1e-4)\n        model.eval();cut=n//2;changed=seq.clone();changed[:,cut:]=(changed[:,cut:]+7)%model.config.vocab\n        altered=model(changed).float()\n        errors[\'futureMaxAbs\']=float((full[:,:cut]-altered[:,:cut]).abs().max())\n        checks[\'future_cannot_change_past\']=torch.allclose(full[:,:cut],altered[:,:cut],atol=1e-3,rtol=1e-4)\n        prefix=model(seq[:,:cut]).float()\n        errors[\'prefixMaxAbs\']=float((full[:,cut-1]-prefix[:,-1]).abs().max())\n        checks[\'prefix_and_teacher_forced_agree\']=torch.allclose(full[:,cut-1],prefix[:,-1],atol=1e-3,rtol=1e-4)\n        checks[\'finite_logits\']=bool(torch.isfinite(full).all())\n        return dict(format=\'nebula-v30-device-audit-1\',passed=all(checks.values()),checks=checks,maxErrors=errors,\n            device=str(device),scope=\'Small actual checkpoint/device sample; does not certify factual knowledge or all training behavior.\')\n    finally:\n        model.train(original_mode);torch.backends.cuda.matmul.allow_tf32=tf32\n', 'vd2_audit.py': '"""Read-only, bounded data inspection. Reports counts, never text or approvals."""\nimport collections,json,re\nfrom vd2_data import texts,clean,category,sha,source_split\nfrom vd2_hub import Hub\n\ndef audit_sources(inventory,token=None,max_bytes=512_000_000,max_object_bytes=128_000_000,reader=None):\n    if reader is None:\n        import requests\n        hub=object.__new__(Hub);hub.token=token or \'\';hub.session=requests.Session()\n        reader=hub.read_entry # Does not construct Hub, create a repo or write anything.\n    groups={}\n    for source in inventory[\'candidates\']:\n        groups.setdefault((source[\'entry\'][\'sha256\'],source[\'format\']),[]).append(source)\n    records={};reports=[];used=0\n    for aliases in sorted(groups.values(),key=lambda a:all(source_split(s)!=\'validation\' for s in a)):\n        source=aliases[0];size=source[\'entry\'][\'bytes\']\n        split=\'validation\' if any(source_split(s)==\'validation\' for s in aliases) else \'train\'\n        report=dict(repo=source[\'repo\'],revision=source[\'revision\'],key=source[\'key\'],aliases=len(aliases),split=split,bytes=size,approved=False)\n        if source[\'format\']!=\'jsonl\':report[\'status\']=\'not_inspected_encoded_tokens\'\n        elif size>max_object_bytes or used+size>max_bytes:report[\'status\']=\'not_inspected_byte_budget\'\n        else:\n            used+=size\n            try:\n                blob=reader(source[\'repo\'],source[\'revision\'],source[\'entry\'])\n                counts=collections.Counter();rejected=0;pending={}\n                for value in texts(blob,\'jsonl\'):\n                    text=clean(value)\n                    if text is None:rejected+=1;continue\n                    fingerprint=sha(re.sub(r\'\\s+\',\' \',text).encode());kind=category(text);counts[kind]+=1\n                    target=\'validation\' if split==\'validation\' or int(fingerprint[:8],16)%50==0 else \'train\'\n                    if fingerprint not in pending or target==\'validation\':pending[fingerprint]=(target,kind)\n                # A malformed object must not contribute a partially parsed corpus.\n                for fingerprint,value in pending.items():\n                    if fingerprint not in records or value[0]==\'validation\':records[fingerprint]=value\n                report.update(status=\'inspected\',counts=dict(counts),rejected=rejected)\n                del blob\n            except Exception as error:\n                # Exception messages may contain signed URLs or example text.\n                report.update(status=\'inspection_failed\',errorType=type(error).__name__)\n        reports.append(report)\n        print(json.dumps({\'vd2SourceAudit\':report}),flush=True)\n    totals=collections.Counter(\'/\'.join(v) for v in records.values())\n    result=dict(readOnly=True,objectsMoved=False,trainingStarted=False,sourcesApproved=False,bytesRead=used,\n                inspectedObjects=sum(r[\'status\']==\'inspected\' for r in reports),totalObjects=len(reports),\n                uniqueInspectedDocuments=len(records),counts=dict(totals),minimumCodeDocuments=5000,\n                inspectedCodeFloorMet=totals[\'train/code\']>=5000,\n                note=\'Code counts are heuristic; provenance, licenses and uninspected objects still require review.\',sources=reports)\n    print(json.dumps({\'vd2AuditSummary\':{k:v for k,v in result.items() if k!=\'sources\'}}),flush=True)\n    return result\n', 'vd2_public_code.py': '"""Pinned, licensed Python sources -> private D2 corpus. Never execute source code."""\nimport ast,copy,gzip,hashlib,io,json,tarfile,tokenize\nfrom pathlib import PurePosixPath\nfrom vd2_data import Corpus,sha,SPECIAL\nfrom vd2_lab import DEFAULT,engine_hash\n\ndef fetch_archive(source,max_bytes=80_000_000):\n    import requests\n    repo=source[\'repo\'];revision=source[\'revision\']\n    import re\n    if not re.fullmatch(r\'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\',repo) or not re.fullmatch(r\'[a-f0-9]{40}\',revision):raise ValueError(\'Pin an exact GitHub source commit\')\n    data=bytearray()\n    with requests.get(f\'https://codeload.github.com/{repo}/tar.gz/{revision}\',stream=True,timeout=(30,180)) as response:\n        response.raise_for_status()\n        for chunk in response.iter_content(1024*1024):\n            data.extend(chunk)\n            if len(data)>max_bytes:raise ValueError(\'Source archive exceeds RAM budget\')\n    return bytes(data)\n\ndef collect(sources,config=DEFAULT,fetcher=fetch_archive):\n    corpus=Corpus(config[\'maxCorpusBytes\']);provenance=[];skipped={}\n    for spec in sources:\n        if not spec.get(\'approved\') or not spec.get(\'licenseText\') or spec.get(\'origin\')!=\'public-licensed\':raise ValueError(\'Explicitly reviewed public source and full license required\')\n        archive=fetcher(spec);archive_hash=sha(archive)\n        if spec.get(\'archiveSha256\') and archive_hash!=spec[\'archiveSha256\']:raise ValueError(\'Pinned source archive checksum changed\')\n        count=0\n        with tarfile.open(fileobj=io.BytesIO(archive),mode=\'r:gz\') as tar:\n            members=tar.getmembers();roots={m.name.split(\'/\')[0] for m in members}\n            if len(roots)!=1:raise ValueError(\'Unexpected archive root\')\n            root=next(iter(roots))+\'/\'\n            notice=tar.extractfile(root+spec[\'licensePath\']).read().decode(\'utf-8\')\n            if notice!=spec[\'licenseText\']:raise ValueError(\'License text differs from reviewed commit\')\n            for member in members:\n                name=member.name[len(root):];path=PurePosixPath(name)\n                if not member.isfile() or \'..\' in path.parts or not name.endswith(\'.py\') or not name.startswith(spec.get(\'prefix\',\'\')):continue\n                if any(p.lower() in [\'vendor\',\'vendored\',\'_vendor\',\'third_party\',\'third-party\',\'node_modules\',\'data\',\'datafiles\'] for p in path.parts):continue\n                if not 60<=member.size<=150000:continue\n                try:\n                    raw=tar.extractfile(member).read();encoding,_=tokenize.detect_encoding(io.BytesIO(raw).readline);text=raw.decode(encoding)\n                    ast.parse(text,filename=name) # Syntax check only. No imports or execution.\n                except (SyntaxError,UnicodeError,LookupError,ValueError,RecursionError):skipped[\'syntax-or-encoding\']=skipped.get(\'syntax-or-encoding\',0)+1;continue\n                # Whole directory groups stay together; content dedup also moves overlaps to validation.\n                group=\'/\'.join(path.parts[:2]) if spec[\'repo\']!=\'python/cpython\' else \'/\'.join(path.parts[:3])\n                split=\'validation\' if int(sha((spec[\'repo\']+\'/\'+group).encode())[:8],16)%20==0 else \'train\'\n                source=dict(repo=\'github:\'+spec[\'repo\'],revision=spec[\'revision\'],key=name,license=spec[\'license\'],origin=\'public-licensed\',approved=True,split=split)\n                corpus.add(text,source);count+=1\n        provenance.append({**spec,\'archiveSha256\':archive_hash,\'syntaxCheckedPythonFiles\':count})\n        print(json.dumps({\'vd2PublicSource\':spec[\'repo\'],\'filesChecked\':count,**corpus.summary()}),flush=True)\n        del archive\n    return corpus,provenance,skipped\n\ndef prepare_public(repo,token,sources,config=DEFAULT):\n    from tokenizers import Tokenizer,models,pre_tokenizers,decoders,trainers\n    from vd2_hub import Hub\n    hub=Hub(repo,token);existing=hub.state()\n    signature=sha(json.dumps(dict(config=config,sources=sources,engine=engine_hash(),publicPipeline=sha(open(__file__,\'rb\').read())),sort_keys=True).encode())\n    if existing:\n        if existing.get(\'signature\')!=signature:raise ValueError(\'Changed public corpus/config: choose a new private destination\')\n        if existing.get(\'dataReady\'):return existing\n    corpus,provenance,skipped=collect(sources,config)\n    summary=corpus.summary()\n    if summary[\'counts\'].get(\'train/code\',0)<config[\'minCodeDocuments\']:raise ValueError(\'Not enough unique syntax-checked coding documents\')\n    if summary[\'counts\'].get(\'validation/code\',0)<50:raise ValueError(\'Insufficient independent coding validation\')\n    tokenizer=Tokenizer(models.BPE(unk_token=None));tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False);tokenizer.decoder=decoders.ByteLevel()\n    tokenizer.train_from_iterator((r[\'text\'] for r in corpus.records.values() if r[\'split\']==\'train\'),trainers.BpeTrainer(vocab_size=config[\'model\'][\'vocab\'],special_tokens=SPECIAL,initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))\n    if tokenizer.get_vocab_size()!=config[\'model\'][\'vocab\']:raise ValueError(\'Target vocabulary not reached\')\n    state=dict(format=\'nebula-vd2-code-1\',signature=signature,engine=engine_hash(),config=config,dataReady=True,updates=0,qualityApproved=False,websiteUpdated=False,summary=summary,sources=provenance,skipped=skipped,\n               corpusScope=\'Python code foundation pilot; no instruction tuning or user chats; directory/content holdouts. Syntax checking does not establish correctness.\')\n    notices=json.dumps(dict(sources=provenance,changes=\'Filtered Python files, normalized newlines, deduplicated and split for a private foundation-training corpus. No source executed.\'),ensure_ascii=False).encode()\n    return hub.publish(state,{\'corpus\':gzip.compress(corpus.payload()),\'tokenizer\':tokenizer.to_str().encode(),\'sourceNotices\':gzip.compress(notices)})\n', 'compare_owned_models.py': '"""Same bounded tasks for owned byte-100M and subword-250M checkpoints. No code execution."""\nimport ast,gc,io,json,re,time\n\ndef suite():\n    cases=[]\n    for a,b in [(28,47),(63,19),(107,36),(84,29),(17,8),(91,14),(56,37),(125,42)]:\n        cases.append(dict(category=\'arithmetic\',messages=[dict(role=\'user\',content=f\'What is {a} + {b}? Reply with only the number.\')],number=a+b))\n    for color,obj in [(\'mauve\',\'lantern\'),(\'cobalt\',\'parcel\'),(\'amber\',\'ribbon\'),(\'teal\',\'vase\'),(\'violet\',\'button\'),(\'silver\',\'cup\')]:\n        cases.append(dict(category=\'extraction\',messages=[dict(role=\'user\',content=f\'The {obj} is {color}. What color is the {obj}? Reply with only the color.\')],word=color))\n    for name in [\'Zorivelle\',\'Pemadora\',\'Kivonelle\',\'Talumira\']:\n        cases.append(dict(category=\'context\',messages=[dict(role=\'user\',content=f\'My name is {name}.\'),dict(role=\'assistant\',content=f\'Hello, {name}.\'),dict(role=\'user\',content=\'What is my name? Reply with only my name.\')],word=name))\n    for name,task,op in [(\'total_lumens\',\'returns sum(values)\',\'sum\'),(\'largest_pebble\',\'returns max(values)\',\'max\'),(\'smallest_pebble\',\'returns min(values)\',\'min\'),(\'count_pebbles\',\'returns len(values)\',\'len\')]:\n        cases.append(dict(category=\'coding_static\',messages=[dict(role=\'user\',content=f\'Write Python function {name}(values) that {task}. Return only Python code.\')],function=name,call=op))\n    for prompt in [\'Explain why leaves need sunlight in two sentences.\',\'Write a friendly two-sentence invitation to a picnic.\',\'Explain what a Python list is with a small example.\',\'I have ten minutes to organize my desk. Give three useful steps.\']:\n        cases.append(dict(category=\'qualitative\',messages=[dict(role=\'user\',content=prompt)]))\n    return cases\n\ndef grade(case,text):\n    if case[\'category\']==\'qualitative\':return None\n    if \'number\' in case:\n        numbers=re.findall(r\'(?<![\\w.])-?\\d+(?![\\w.])\',text)\n        return len(numbers)==1 and int(numbers[0])==case[\'number\']\n    if \'word\'in case:return re.search(r\'\\b\'+re.escape(case[\'word\'])+r\'\\b\',text,re.I) is not None\n    code=text.strip()\n    if \'```\' in code:\n        match=re.search(r\'```(?:python|py)?\\s*\\n(.*?)```\',code,re.S)\n        if not match:return False\n        code=match.group(1)\n    try:tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError):return False\n    for node in tree.body:\n        if isinstance(node,ast.FunctionDef) and node.name==case[\'function\']:\n            return any(isinstance(n,ast.Return) and isinstance(n.value,ast.Call) and isinstance(n.value.func,ast.Name) and n.value.func.id==case[\'call\'] and len(n.value.args)==1 and isinstance(n.value.args[0],ast.Name) and n.value.args[0].id==\'values\' for n in ast.walk(node))\n    return False\n\ndef compare(specs,token=\'\',max_new_tokens=64,read_entry=None):\n    import torch\n    from tokenizers import Tokenizer\n    from vd2_hub import Hub\n    from v30_expanded_model import Model,Config,encode_chat\n    import requests\n    device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\';torch.set_num_threads(2);reports=[];tasks=suite()\n    reader=object.__new__(Hub);reader.token=token;reader.session=requests.Session();read_entry=read_entry or reader.read_entry\n    for spec in specs:\n        label=spec[\'label\'];state=spec[\'state\'];key=spec[\'checkpointKey\'];started=time.monotonic()\n        print(json.dumps({\'comparisonLoading\':label,\'checkpoint\':state[key][\'sha256\'],\'bytes\':state[key][\'bytes\']}),flush=True)\n        blob=read_entry(spec[\'repo\'],spec[\'revision\'],state[key]);saved=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True);del blob\n        byte=spec[\'encoding\']==\'bytes\'\n        if byte:\n            c=Config(vocab=262,**saved[\'config\']);weights=saved[\'state\'];tokenizer=None\n        else:\n            c=Config(**saved[\'config\']);weights=saved[\'model\'];tokenizer=Tokenizer.from_str(read_entry(spec[\'repo\'],spec[\'revision\'],state[\'tokenizer\']).decode())\n        model=Model(c);model.load_state_dict(weights);del weights,saved;gc.collect()\n        model.set_attention_backend(\'math\');model.to(device).eval();count=sum(p.numel() for p in model.parameters());rows=[]\n        for index,case in enumerate(tasks):\n            if byte:\n                roles={\'system\':258,\'user\':259,\'assistant\':260};ids=[256]\n                for m in case[\'messages\']:ids+=[roles[m[\'role\']]]+list(m[\'content\'].encode())+[257]\n                ids+=[260];eos=257\n            else:ids=encode_chat(tokenizer,case[\'messages\'],True)[0];eos=2\n            out=[];begin=time.monotonic()\n            budget=128 if case[\'category\']==\'coding_static\' else max_new_tokens\n            with torch.inference_mode():\n                for _ in range(min(budget,c.context-len(ids))):\n                    logits=model(torch.tensor([ids+out],device=device))[0,-1].float()\n                    if not torch.isfinite(logits).all():raise ValueError(\'Nonfinite inference logits\')\n                    if byte:logits[256]=-float(\'inf\');logits[258:]=-float(\'inf\')\n                    else:logits[[0,1,3,4,5,6,7,8]]=-float(\'inf\')\n                    next_token=int(logits.argmax())\n                    if next_token==eos:break\n                    out.append(next_token)\n            text=bytes(out).decode(\'utf-8\',errors=\'replace\') if byte else tokenizer.decode(out,skip_special_tokens=True)\n            row=dict(index=index,category=case[\'category\'],messages=case[\'messages\'],reply=text,passed=grade(case,text),seconds=round(time.monotonic()-begin,3),generatedTokens=len(out));rows.append(row)\n            print(json.dumps({\'comparisonCase\':label,**row}),flush=True)\n        categories={}\n        for category in [\'arithmetic\',\'extraction\',\'context\',\'coding_static\']:\n            chosen=[r for r in rows if r[\'category\']==category];categories[category]=dict(passed=sum(r[\'passed\'] for r in chosen),cases=len(chosen))\n        report=dict(model=label,parameters=count,device=device,repo=spec[\'repo\'],revision=spec[\'revision\'],checkpoint=state[key][\'sha256\'],categories=categories,passed=sum(r[\'passed\'] is True for r in rows),scoredCases=sum(r[\'passed\'] is not None for r in rows),qualitativeCases=4,rows=rows,seconds=round(time.monotonic()-started,2),scope=\'Small task check, not general intelligence. Coding checks syntax/structure only; no generated code executed. No checkpoint promotion or training on these tasks.\')\n        reports.append(report);print(json.dumps({\'comparisonSummary\':{k:v for k,v in report.items() if k!=\'rows\'}}),flush=True)\n        del model,tokenizer;gc.collect();torch.cuda.empty_cache()\n    return reports\n'}
code_root=Path(tempfile.gettempdir())/'nebula-vd2-code'
code_root.mkdir(exist_ok=True)
for name,source in SOURCES.items():(code_root/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(code_root))
for name in SOURCES:sys.modules.pop(Path(name).stem,None)
from vd2_lab import inventory,prepare,train,DEFAULT
from vd2_data import parameter_count
print({'preparedParameters':parameter_count(),'trainingStarted':False,'websiteUpdated':False})


In [ ]:
RUN_COMPARISON=True
PREPARE_D2=True
START_D2_TRAINING=True
OUTPUT_REPO='iiidevv/nebula-vd2-python-1b-20261026'
SESSION_HOURS=6.0
RESERVE_MINUTES=15
CONFIG=__import__('copy').deepcopy(DEFAULT)
PUBLIC_SOURCES=[{'repo': 'python/cpython', 'revision': '3acc941f2b7288b212a4163ab21aa79faff9cd13', 'prefix': 'Lib/', 'licensePath': 'LICENSE', 'license': 'PSF-2.0 and included notices', 'licenseText': 'A. HISTORY OF THE SOFTWARE\n==========================\n\nPython was created in the early 1990s by Guido van Rossum at Stichting\nMathematisch Centrum (CWI, see https://www.cwi.nl) in the Netherlands\nas a successor of a language called ABC.  Guido remains Python\'s\nprincipal author, although it includes many contributions from others.\n\nIn 1995, Guido continued his work on Python at the Corporation for\nNational Research Initiatives (CNRI, see https://www.cnri.reston.va.us)\nin Reston, Virginia where he released several versions of the\nsoftware.\n\nIn May 2000, Guido and the Python core development team moved to\nBeOpen.com to form the BeOpen PythonLabs team.  In October of the same\nyear, the PythonLabs team moved to Digital Creations, which became\nZope Corporation.  In 2001, the Python Software Foundation (PSF, see\nhttps://www.python.org/psf/) was formed, a non-profit organization\ncreated specifically to own Python-related Intellectual Property.\nZope Corporation was a sponsoring member of the PSF.\n\nAll Python releases are Open Source (see https://opensource.org for\nthe Open Source Definition).  Historically, most, but not all, Python\nreleases have also been GPL-compatible; the table below summarizes\nthe various releases.\n\n    Release         Derived     Year        Owner       GPL-\n                    from                                compatible? (1)\n\n    0.9.0 thru 1.2              1991-1995   CWI         yes\n    1.3 thru 1.5.2  1.2         1995-1999   CNRI        yes\n    1.6             1.5.2       2000        CNRI        no\n    2.0             1.6         2000        BeOpen.com  no\n    1.6.1           1.6         2001        CNRI        yes (2)\n    2.1             2.0+1.6.1   2001        PSF         no\n    2.0.1           2.0+1.6.1   2001        PSF         yes\n    2.1.1           2.1+2.0.1   2001        PSF         yes\n    2.1.2           2.1.1       2002        PSF         yes\n    2.1.3           2.1.2       2002        PSF         yes\n    2.2 and above   2.1.1       2001-now    PSF         yes\n\nFootnotes:\n\n(1) GPL-compatible doesn\'t mean that we\'re distributing Python under\n    the GPL.  All Python licenses, unlike the GPL, let you distribute\n    a modified version without making your changes open source.  The\n    GPL-compatible licenses make it possible to combine Python with\n    other software that is released under the GPL; the others don\'t.\n\n(2) According to Richard Stallman, 1.6.1 is not GPL-compatible,\n    because its license has a choice of law clause.  According to\n    CNRI, however, Stallman\'s lawyer has told CNRI\'s lawyer that 1.6.1\n    is "not incompatible" with the GPL.\n\nThanks to the many outside volunteers who have worked under Guido\'s\ndirection to make these releases possible.\n\n\nB. TERMS AND CONDITIONS FOR ACCESSING OR OTHERWISE USING PYTHON\n===============================================================\n\nPython software and documentation are licensed under the\nPython Software Foundation License Version 2.\n\nStarting with Python 3.8.6, examples, recipes, and other code in\nthe documentation are dual licensed under the PSF License Version 2\nand the Zero-Clause BSD license.\n\nSome software incorporated into Python is under different licenses.\nThe licenses are listed with code falling under that license.\n\n\nPYTHON SOFTWARE FOUNDATION LICENSE VERSION 2\n--------------------------------------------\n\n1. This LICENSE AGREEMENT is between the Python Software Foundation\n("PSF"), and the Individual or Organization ("Licensee") accessing and\notherwise using this software ("Python") in source or binary form and\nits associated documentation.\n\n2. Subject to the terms and conditions of this License Agreement, PSF hereby\ngrants Licensee a nonexclusive, royalty-free, world-wide license to reproduce,\nanalyze, test, perform and/or display publicly, prepare derivative works,\ndistribute, and otherwise use Python alone or in any derivative version,\nprovided, however, that PSF\'s License Agreement and PSF\'s notice of copyright,\ni.e., "Copyright (c) 2001 Python Software Foundation; All Rights Reserved"\nare retained in Python alone or in any derivative version prepared by Licensee.\n\n3. In the event Licensee prepares a derivative work that is based on\nor incorporates Python or any part thereof, and wants to make\nthe derivative work available to others as provided herein, then\nLicensee hereby agrees to include in any such work a brief summary of\nthe changes made to Python.\n\n4. PSF is making Python available to Licensee on an "AS IS"\nbasis.  PSF MAKES NO REPRESENTATIONS OR WARRANTIES, EXPRESS OR\nIMPLIED.  BY WAY OF EXAMPLE, BUT NOT LIMITATION, PSF MAKES NO AND\nDISCLAIMS ANY REPRESENTATION OR WARRANTY OF MERCHANTABILITY OR FITNESS\nFOR ANY PARTICULAR PURPOSE OR THAT THE USE OF PYTHON WILL NOT\nINFRINGE ANY THIRD PARTY RIGHTS.\n\n5. PSF SHALL NOT BE LIABLE TO LICENSEE OR ANY OTHER USERS OF PYTHON\nFOR ANY INCIDENTAL, SPECIAL, OR CONSEQUENTIAL DAMAGES OR LOSS AS\nA RESULT OF MODIFYING, DISTRIBUTING, OR OTHERWISE USING PYTHON,\nOR ANY DERIVATIVE THEREOF, EVEN IF ADVISED OF THE POSSIBILITY THEREOF.\n\n6. This License Agreement will automatically terminate upon a material\nbreach of its terms and conditions.\n\n7. Nothing in this License Agreement shall be deemed to create any\nrelationship of agency, partnership, or joint venture between PSF and\nLicensee.  This License Agreement does not grant permission to use PSF\ntrademarks or trade name in a trademark sense to endorse or promote\nproducts or services of Licensee, or any third party.\n\n8. By copying, installing or otherwise using Python, Licensee\nagrees to be bound by the terms and conditions of this License\nAgreement.\n\n\nBEOPEN.COM LICENSE AGREEMENT FOR PYTHON 2.0\n-------------------------------------------\n\nBEOPEN PYTHON OPEN SOURCE LICENSE AGREEMENT VERSION 1\n\n1. This LICENSE AGREEMENT is between BeOpen.com ("BeOpen"), having an\noffice at 160 Saratoga Avenue, Santa Clara, CA 95051, and the\nIndividual or Organization ("Licensee") accessing and otherwise using\nthis software in source or binary form and its associated\ndocumentation ("the Software").\n\n2. Subject to the terms and conditions of this BeOpen Python License\nAgreement, BeOpen hereby grants Licensee a non-exclusive,\nroyalty-free, world-wide license to reproduce, analyze, test, perform\nand/or display publicly, prepare derivative works, distribute, and\notherwise use the Software alone or in any derivative version,\nprovided, however, that the BeOpen Python License is retained in the\nSoftware, alone or in any derivative version prepared by Licensee.\n\n3. BeOpen is making the Software available to Licensee on an "AS IS"\nbasis.  BEOPEN MAKES NO REPRESENTATIONS OR WARRANTIES, EXPRESS OR\nIMPLIED.  BY WAY OF EXAMPLE, BUT NOT LIMITATION, BEOPEN MAKES NO AND\nDISCLAIMS ANY REPRESENTATION OR WARRANTY OF MERCHANTABILITY OR FITNESS\nFOR ANY PARTICULAR PURPOSE OR THAT THE USE OF THE SOFTWARE WILL NOT\nINFRINGE ANY THIRD PARTY RIGHTS.\n\n4. BEOPEN SHALL NOT BE LIABLE TO LICENSEE OR ANY OTHER USERS OF THE\nSOFTWARE FOR ANY INCIDENTAL, SPECIAL, OR CONSEQUENTIAL DAMAGES OR LOSS\nAS A RESULT OF USING, MODIFYING OR DISTRIBUTING THE SOFTWARE, OR ANY\nDERIVATIVE THEREOF, EVEN IF ADVISED OF THE POSSIBILITY THEREOF.\n\n5. This License Agreement will automatically terminate upon a material\nbreach of its terms and conditions.\n\n6. This License Agreement shall be governed by and interpreted in all\nrespects by the law of the State of California, excluding conflict of\nlaw provisions.  Nothing in this License Agreement shall be deemed to\ncreate any relationship of agency, partnership, or joint venture\nbetween BeOpen and Licensee.  This License Agreement does not grant\npermission to use BeOpen trademarks or trade names in a trademark\nsense to endorse or promote products or services of Licensee, or any\nthird party.  As an exception, the "BeOpen Python" logos available at\nhttp://www.pythonlabs.com/logos.html may be used according to the\npermissions granted on that web page.\n\n7. By copying, installing or otherwise using the software, Licensee\nagrees to be bound by the terms and conditions of this License\nAgreement.\n\n\nCNRI LICENSE AGREEMENT FOR PYTHON 1.6.1\n---------------------------------------\n\n1. This LICENSE AGREEMENT is between the Corporation for National\nResearch Initiatives, having an office at 1895 Preston White Drive,\nReston, VA 20191 ("CNRI"), and the Individual or Organization\n("Licensee") accessing and otherwise using Python 1.6.1 software in\nsource or binary form and its associated documentation.\n\n2. Subject to the terms and conditions of this License Agreement, CNRI\nhereby grants Licensee a nonexclusive, royalty-free, world-wide\nlicense to reproduce, analyze, test, perform and/or display publicly,\nprepare derivative works, distribute, and otherwise use Python 1.6.1\nalone or in any derivative version, provided, however, that CNRI\'s\nLicense Agreement and CNRI\'s notice of copyright, i.e., "Copyright (c)\n1995-2001 Corporation for National Research Initiatives; All Rights\nReserved" are retained in Python 1.6.1 alone or in any derivative\nversion prepared by Licensee.  Alternately, in lieu of CNRI\'s License\nAgreement, Licensee may substitute the following text (omitting the\nquotes): "Python 1.6.1 is made available subject to the terms and\nconditions in CNRI\'s License Agreement.  This Agreement together with\nPython 1.6.1 may be located on the internet using the following\nunique, persistent identifier (known as a handle): 1895.22/1013.  This\nAgreement may also be obtained from a proxy server on the internet\nusing the following URL: http://hdl.handle.net/1895.22/1013".\n\n3. In the event Licensee prepares a derivative work that is based on\nor incorporates Python 1.6.1 or any part thereof, and wants to make\nthe derivative work available to others as provided herein, then\nLicensee hereby agrees to include in any such work a brief summary of\nthe changes made to Python 1.6.1.\n\n4. CNRI is making Python 1.6.1 available to Licensee on an "AS IS"\nbasis.  CNRI MAKES NO REPRESENTATIONS OR WARRANTIES, EXPRESS OR\nIMPLIED.  BY WAY OF EXAMPLE, BUT NOT LIMITATION, CNRI MAKES NO AND\nDISCLAIMS ANY REPRESENTATION OR WARRANTY OF MERCHANTABILITY OR FITNESS\nFOR ANY PARTICULAR PURPOSE OR THAT THE USE OF PYTHON 1.6.1 WILL NOT\nINFRINGE ANY THIRD PARTY RIGHTS.\n\n5. CNRI SHALL NOT BE LIABLE TO LICENSEE OR ANY OTHER USERS OF PYTHON\n1.6.1 FOR ANY INCIDENTAL, SPECIAL, OR CONSEQUENTIAL DAMAGES OR LOSS AS\nA RESULT OF MODIFYING, DISTRIBUTING, OR OTHERWISE USING PYTHON 1.6.1,\nOR ANY DERIVATIVE THEREOF, EVEN IF ADVISED OF THE POSSIBILITY THEREOF.\n\n6. This License Agreement will automatically terminate upon a material\nbreach of its terms and conditions.\n\n7. This License Agreement shall be governed by the federal\nintellectual property law of the United States, including without\nlimitation the federal copyright law, and, to the extent such\nU.S. federal law does not apply, by the law of the Commonwealth of\nVirginia, excluding Virginia\'s conflict of law provisions.\nNotwithstanding the foregoing, with regard to derivative works based\non Python 1.6.1 that incorporate non-separable material that was\npreviously distributed under the GNU General Public License (GPL), the\nlaw of the Commonwealth of Virginia shall govern this License\nAgreement only as to issues arising under or with respect to\nParagraphs 4, 5, and 7 of this License Agreement.  Nothing in this\nLicense Agreement shall be deemed to create any relationship of\nagency, partnership, or joint venture between CNRI and Licensee.  This\nLicense Agreement does not grant permission to use CNRI trademarks or\ntrade name in a trademark sense to endorse or promote products or\nservices of Licensee, or any third party.\n\n8. By clicking on the "ACCEPT" button where indicated, or by copying,\ninstalling or otherwise using Python 1.6.1, Licensee agrees to be\nbound by the terms and conditions of this License Agreement.\n\n        ACCEPT\n\n\nCWI LICENSE AGREEMENT FOR PYTHON 0.9.0 THROUGH 1.2\n--------------------------------------------------\n\nCopyright (c) 1991 - 1995, Stichting Mathematisch Centrum Amsterdam,\nThe Netherlands.  All rights reserved.\n\nPermission to use, copy, modify, and distribute this software and its\ndocumentation for any purpose and without fee is hereby granted,\nprovided that the above copyright notice appear in all copies and that\nboth that copyright notice and this permission notice appear in\nsupporting documentation, and that the name of Stichting Mathematisch\nCentrum or CWI not be used in advertising or publicity pertaining to\ndistribution of the software without specific, written prior\npermission.\n\nSTICHTING MATHEMATISCH CENTRUM DISCLAIMS ALL WARRANTIES WITH REGARD TO\nTHIS SOFTWARE, INCLUDING ALL IMPLIED WARRANTIES OF MERCHANTABILITY AND\nFITNESS, IN NO EVENT SHALL STICHTING MATHEMATISCH CENTRUM BE LIABLE\nFOR ANY SPECIAL, INDIRECT OR CONSEQUENTIAL DAMAGES OR ANY DAMAGES\nWHATSOEVER RESULTING FROM LOSS OF USE, DATA OR PROFITS, WHETHER IN AN\nACTION OF CONTRACT, NEGLIGENCE OR OTHER TORTIOUS ACTION, ARISING OUT\nOF OR IN CONNECTION WITH THE USE OR PERFORMANCE OF THIS SOFTWARE.\n\nZERO-CLAUSE BSD LICENSE FOR CODE IN THE PYTHON DOCUMENTATION\n----------------------------------------------------------------------\n\nPermission to use, copy, modify, and/or distribute this software for any\npurpose with or without fee is hereby granted.\n\nTHE SOFTWARE IS PROVIDED "AS IS" AND THE AUTHOR DISCLAIMS ALL WARRANTIES WITH\nREGARD TO THIS SOFTWARE INCLUDING ALL IMPLIED WARRANTIES OF MERCHANTABILITY\nAND FITNESS. IN NO EVENT SHALL THE AUTHOR BE LIABLE FOR ANY SPECIAL, DIRECT,\nINDIRECT, OR CONSEQUENTIAL DAMAGES OR ANY DAMAGES WHATSOEVER RESULTING FROM\nLOSS OF USE, DATA OR PROFITS, WHETHER IN AN ACTION OF CONTRACT, NEGLIGENCE OR\nOTHER TORTIOUS ACTION, ARISING OUT OF OR IN CONNECTION WITH THE USE OR\nPERFORMANCE OF THIS SOFTWARE.\n', 'approved': True, 'origin': 'public-licensed', 'archiveSha256': 'a30991b0a922da1d6fac27608576e8cd04c2326dde3a4f2f9fd109ef912a85f9'}, {'repo': 'django/django', 'revision': 'e9414948e9660ee4c167eebb187710ee3031d9d8', 'prefix': '', 'licensePath': 'LICENSE', 'license': 'BSD-3-Clause', 'licenseText': 'Copyright (c) Django Software Foundation and individual contributors.\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without modification,\nare permitted provided that the following conditions are met:\n\n    1. Redistributions of source code must retain the above copyright notice,\n       this list of conditions and the following disclaimer.\n\n    2. Redistributions in binary form must reproduce the above copyright\n       notice, this list of conditions and the following disclaimer in the\n       documentation and/or other materials provided with the distribution.\n\n    3. Neither the name of Django nor the names of its contributors may be used\n       to endorse or promote products derived from this software without\n       specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES\n(INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON\nANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n(INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'approved': True, 'origin': 'public-licensed', 'archiveSha256': 'fac59d69c1fb86efe294fa58e8c23c4b6d4e0fa17d44e094e992c8b43d66273e'}, {'repo': 'pytest-dev/pytest', 'revision': '41d12cdbb4a35a7877768d93d8dc50a9117a8d8c', 'prefix': '', 'licensePath': 'LICENSE', 'license': 'MIT', 'licenseText': 'The MIT License (MIT)\n\nCopyright (c) 2004 Holger Krekel and others\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of\nthis software and associated documentation files (the "Software"), to deal in\nthe Software without restriction, including without limitation the rights to\nuse, copy, modify, merge, publish, distribute, sublicense, and/or sell copies\nof the Software, and to permit persons to whom the Software is furnished to do\nso, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'approved': True, 'origin': 'public-licensed', 'archiveSha256': '94f78cbab3fec9481bbb0a3ec73e2dfb0136ea8b89b0c4c17228bcdf4b15a4f5'}, {'repo': 'sympy/sympy', 'revision': 'cd86f04255d9411f11f32108cfa4ef773e4f401b', 'prefix': 'sympy/', 'licensePath': 'LICENSE', 'license': 'BSD-3-Clause / MIT and included notices', 'licenseText': 'Copyright (c) 2006-2023 SymPy Development Team\n\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n  a. Redistributions of source code must retain the above copyright notice,\n     this list of conditions and the following disclaimer.\n  b. Redistributions in binary form must reproduce the above copyright\n     notice, this list of conditions and the following disclaimer in the\n     documentation and/or other materials provided with the distribution.\n  c. Neither the name of SymPy nor the names of its contributors\n     may be used to endorse or promote products derived from this software\n     without specific prior written permission.\n\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE\nARE DISCLAIMED. IN NO EVENT SHALL THE REGENTS OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT\nLIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY\nOUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH\nDAMAGE.\n\n--------------------------------------------------------------------------------\n\nPatches that were taken from the Diofant project (https://github.com/diofant/diofant)\nare licensed as:\n\nCopyright (c) 2006-2018 SymPy Development Team,\n              2013-2023 Sergey B Kirpichev\n\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n  a. Redistributions of source code must retain the above copyright notice,\n     this list of conditions and the following disclaimer.\n  b. Redistributions in binary form must reproduce the above copyright\n     notice, this list of conditions and the following disclaimer in the\n     documentation and/or other materials provided with the distribution.\n  c. Neither the name of Diofant or SymPy nor the names of its contributors\n     may be used to endorse or promote products derived from this software\n     without specific prior written permission.\n\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE\nARE DISCLAIMED. IN NO EVENT SHALL THE REGENTS OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT\nLIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY\nOUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH\nDAMAGE.\n\n--------------------------------------------------------------------------------\n\nSubmodules taken from the multipledispatch project (https://github.com/mrocklin/multipledispatch)\nare licensed as:\n\nCopyright (c) 2014 Matthew Rocklin\n\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n  a. Redistributions of source code must retain the above copyright notice,\n     this list of conditions and the following disclaimer.\n  b. Redistributions in binary form must reproduce the above copyright\n     notice, this list of conditions and the following disclaimer in the\n     documentation and/or other materials provided with the distribution.\n  c. Neither the name of multipledispatch nor the names of its contributors\n     may be used to endorse or promote products derived from this software\n     without specific prior written permission.\n\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE\nARE DISCLAIMED. IN NO EVENT SHALL THE REGENTS OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT\nLIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY\nOUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH\nDAMAGE.\n\n--------------------------------------------------------------------------------\n\nThe files under the directory sympy/parsing/autolev/tests/pydy-example-repo\nare directly copied from PyDy project and are licensed as:\n\nCopyright (c) 2009-2023, PyDy Authors\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n* Redistributions of source code must retain the above copyright\n  notice, this list of conditions and the following disclaimer.\n* Redistributions in binary form must reproduce the above copyright\n  notice, this list of conditions and the following disclaimer in the\n  documentation and/or other materials provided with the distribution.\n* Neither the name of this project nor the names of its contributors may be\n  used to endorse or promote products derived from this software without\n  specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL PYDY AUTHORS BE LIABLE FOR ANY DIRECT,\nINDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING,\nBUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE,\nDATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF\nLIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE\nOR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF\nADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n\n--------------------------------------------------------------------------------\n\nThe files under the directory sympy/parsing/latex \nare directly copied from latex2sympy project and are licensed as:\n\nCopyright 2016, latex2sympy\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'approved': True, 'origin': 'public-licensed', 'archiveSha256': '6b69ff846cbbe157f276e06899b5966f883ac77ac4515a48addd7f28390af64b'}]
COMPARISON_SPECS=[{'label': '100M deployed V2.1 source', 'repo': 'iiidevv/nebula-research', 'revision': 'b1bd4ffd41cb79f58f0cb779608f97ed5a75cb31', 'state': {'format': 'nebula-v8-100m-1', 'round': 8, 'rejected': 3, 'bestReport': {'loss': 0.6017579140388745, 'correct': 57, 'cases': 76, 'health': 0.9868421052631579, 'identity': 0.5, 'math': 0.7083333333333334, 'memory': 1.0, 'extraction': 0.9285714285714286, 'familiar': 0.3333333333333333, 'conversation': 0.3333333333333333, 'chatLoss': 1.0239877812564373, 'chatChecks': 5, 'chatCases': 20}, 'workingReport': {'loss': 0.6017579140388745, 'correct': 57, 'cases': 76, 'health': 0.9868421052631579, 'identity': 0.5, 'math': 0.7083333333333334, 'memory': 1.0, 'extraction': 0.9285714285714286, 'familiar': 0.3333333333333333, 'conversation': 0.3333333333333333, 'chatLoss': 1.0239877812564373, 'chatChecks': 5, 'chatCases': 20}, 'baselineReport': {'loss': 0.6017579140388745, 'correct': 57, 'cases': 76, 'health': 0.9868421052631579, 'identity': 0.5, 'math': 0.7083333333333334, 'memory': 1.0, 'extraction': 0.9285714285714286, 'familiar': 0.3333333333333333, 'conversation': 0.3333333333333333, 'chatLoss': 1.0239877812564373, 'chatChecks': 5, 'chatCases': 20}, 'sourceRevision': 'c66600fdf22ba39ab8f98512ae08861c84d20ea6', 'sourceRun': 'experiments/nebula-v7/', 'sourceRound': 2, 'growth': {'originalConfig': {'width': 256, 'heads': 8, 'layers': 10, 'context': 1024}, 'config': {'width': 768, 'heads': 24, 'layers': 14, 'context': 1024}, 'parameters': 100219392, 'forwardPreservedWithinTolerance': True}, 'profile': {'gpus': 2, 'completedSteps': 4, 'worldSize': 2, 'targetsPerSecond': 487, 'peakGpuMemoryBytes': 2463444480}, 'dataSummary': {'format': 'nebula-v7-quality-1', 'conversations': 32338, 'completeConversationExamples': 6198, 'droppedLongIncompleteOrForeignExamples': 389, 'conversationPercent': 80, 'mixedPercent': 20, 'memoryIncluded': True, 'teacherGeneratedSeparately': True, 'dataSha256': '8e4068035dd73081f93e849af151c2cda2e4357840c1975c9ca2bdc46ae53f68'}, 'teacherNextIndex': 2053, 'publicAutoUpdate': False, 'sourceData': {'path': 'experiments/nebula-v8-100m/source-data.pt', 'bytes': 7826357, 'sha256': '23c60d838b82682bb0b7f71312d0740e56d78d6c0712bd29c8080746eacfeb46'}, 'baseline': {'bytes': 400938295, 'sha256': '90d9386332aff512b30b23d1c4aa2b1de689d611c1d3bb20cac400972b966eb2', 'parts': [{'path': 'experiments/nebula-v8-100m/baseline.pt.part000', 'bytes': 134217728, 'sha256': '5b7a15a6cbcdac10729ffa0cbb59611c70d1e479073c88e9a6895297d69b2aa4'}, {'path': 'experiments/nebula-v8-100m/baseline.pt.part001', 'bytes': 134217728, 'sha256': '18e8dee7692aa9f52028982d6eb91bed42fda4a2449cd6322cd07cf0a71802bf'}, {'path': 'experiments/nebula-v8-100m/baseline.pt.part002', 'bytes': 132502839, 'sha256': 'faeba57075ccf77c332f37b951d537bc5400328a802b60310ef2670381c92c63'}]}, 'best': {'bytes': 400938295, 'sha256': '90d9386332aff512b30b23d1c4aa2b1de689d611c1d3bb20cac400972b966eb2', 'parts': [{'path': 'experiments/nebula-v8-100m/best.pt.part000', 'bytes': 134217728, 'sha256': '5b7a15a6cbcdac10729ffa0cbb59611c70d1e479073c88e9a6895297d69b2aa4'}, {'path': 'experiments/nebula-v8-100m/best.pt.part001', 'bytes': 134217728, 'sha256': '18e8dee7692aa9f52028982d6eb91bed42fda4a2449cd6322cd07cf0a71802bf'}, {'path': 'experiments/nebula-v8-100m/best.pt.part002', 'bytes': 132502839, 'sha256': 'faeba57075ccf77c332f37b951d537bc5400328a802b60310ef2670381c92c63'}]}, 'working': {'bytes': 400938295, 'sha256': '90d9386332aff512b30b23d1c4aa2b1de689d611c1d3bb20cac400972b966eb2', 'parts': [{'path': 'experiments/nebula-v8-100m/working.pt.part000', 'bytes': 134217728, 'sha256': '5b7a15a6cbcdac10729ffa0cbb59611c70d1e479073c88e9a6895297d69b2aa4'}, {'path': 'experiments/nebula-v8-100m/working.pt.part001', 'bytes': 134217728, 'sha256': '18e8dee7692aa9f52028982d6eb91bed42fda4a2449cd6322cd07cf0a71802bf'}, {'path': 'experiments/nebula-v8-100m/working.pt.part002', 'bytes': 132502839, 'sha256': 'faeba57075ccf77c332f37b951d537bc5400328a802b60310ef2670381c92c63'}]}, 'data': {'path': 'experiments/nebula-v8-100m/data/v7-chat.jsonl', 'bytes': 13092603, 'sha256': '8e4068035dd73081f93e849af151c2cda2e4357840c1975c9ca2bdc46ae53f68'}, 'evaluationData': {'path': 'experiments/nebula-v8-100m/data/legacy.jsonl', 'bytes': 11262181, 'sha256': '7cd800758355eaded830703cb3f3977480dd5aee8b726db8f643749b9080f2b0'}, 'checks': {'path': 'experiments/nebula-v8-100m/data/checks.json', 'bytes': 16607, 'sha256': '69326993e186d37e8375c84e23596ba64eccf1a5d93e7da19a7c607ed8bd82ac'}, 'qualityAnchor': {'path': 'experiments/nebula-v8-100m/quality-anchor.pt', 'bytes': 32952849, 'sha256': 'fb54811008d166a6c5bb0b3d5a6311af894d9aa6bcee1311b183ee0a54f8a0df'}, 'initialReport': {'path': 'experiments/nebula-v8-100m/reports/baseline.json', 'bytes': 8267, 'sha256': '7d347a4241dbb3338397e97095f7b2548a884d1eaaa084b29bcd9110a7eec46e'}, 'teacherData': {'path': 'experiments/nebula-v8-100m/data/teacher.jsonl', 'bytes': 81044, 'sha256': '9b5071a05c5b13c7231afc7ee1882546a0089bcbcb8f3089862a90e31ed6c974'}, 'report-final': {'path': 'experiments/nebula-v8-100m/reports/0008-final.json', 'bytes': 7687, 'sha256': '707d0ff57f0111593f31e82bdd2ee05f8c19014237939743e5ba82dae8ebb637'}, 'report-lowest-loss': {'path': 'experiments/nebula-v8-100m/reports/0008-lowest-loss.json', 'bytes': 7687, 'sha256': '707d0ff57f0111593f31e82bdd2ee05f8c19014237939743e5ba82dae8ebb637'}, 'latest': {'bytes': 400937861, 'sha256': 'd98ad0f3dc04f270340816536e3ffa386c7ddab7520c7a51ff4f299a0c82ea5a', 'parts': [{'path': 'experiments/nebula-v8-100m/latest.pt.part000', 'bytes': 134217728, 'sha256': 'aee5056c5d22534671e52a32dc7c07545bb5d0e15387b2ae35feeba040dc0e68'}, {'path': 'experiments/nebula-v8-100m/latest.pt.part001', 'bytes': 134217728, 'sha256': '6a1733386bab2a6543ea6fc0009b1abe7c514d9bb0638e3ae3c90d4690abd54c'}, {'path': 'experiments/nebula-v8-100m/latest.pt.part002', 'bytes': 132502405, 'sha256': '81614f78a85d72ffc319bc0916f82f09538d01f85102ea9fd2cff9467c435abf'}]}}, 'checkpointKey': 'best', 'encoding': 'bytes'}, {'label': '100M subword V28 foundation 30000', 'repo': 'iiidevv/nebula-raw-training', 'revision': '6e9739ee3c5700c1b404a52931df329982f43b4a', 'state': {'format': 'nebula-v28-subword-1', 'signature': 'cdae9f006e4fe846240c5ffa5ef5e7eeaca2b00aaf62febf9f35c2a245742159', 'config': {'repo': 'iiidevv/nebula-raw-training', 'rawRun': 'nebula-v18-raw-1gb', 'rawRevision': 'e167fff3bb27c36c25460fcd4c61915a5b7016ed', 'publicRun': 'nebula-v23-longrun', 'publicRevision': 'c21aafa220d33b9787791fcaf7aebf413bc62cf8', 'parentRun': 'nebula-v27-subword', 'parentRevision': '6e46bb0a97a98867a3e6412ad549c4cefb2de84c', 'expectedParentUpdates': 10000, 'trainBytes': 384000000, 'validationBytes': 8000000, 'tokenizerBytes': 16000000, 'sourceMix': {'fineweb': 55, 'cosmopedia': 30, 'textbooks': 10, 'stories': 5}, 'vocab': 8192, 'width': 768, 'heads': 12, 'layers': 13, 'context': 1024, 'seed': 270221, 'foundationUpdates': 30000, 'conversationUpdates': 1000, 'batch': 8, 'microbatch': 1, 'foundationLR': 3e-05, 'conversationLR': 3e-05, 'warmup': 200, 'saveEvery': 100, 'evaluateEvery': 2000}, 'steps': {'foundation': 30000}, 'targets': {'foundation': 245760000}, 'initializedFrom': 'pinned V27 foundation weights; fresh continuation optimizer', 'qualityApproved': False, 'publicWebsiteUpdated': False, 'parentReference': {'repo': 'iiidevv/nebula-raw-training', 'run': 'nebula-v27-subword', 'revision': '6e46bb0a97a98867a3e6412ad549c4cefb2de84c', 'updates': 10000, 'supervisedSubwordTokens': 81920000, 'weightSha256': '86389adbf12fcaa51017003e5aa7386d1921154b2a19091916b709fad8cfbccb', 'tokenizerSha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b'}, 'tokenizerSampleBytes': {'fineweb': 8816863, 'cosmopedia': 4802434, 'textbooks': 1600631, 'stories': 800666}, 'tokenizer': {'bytes': 233872, 'sha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b/tokenizer.json.part000.bin', 'bytes': 233872, 'sha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b'}]}, 'tokenRanges': {'train': {'fineweb': [0, 55273589], 'cosmopedia': [55273589, 81926996], 'textbooks': [81926996, 91246190], 'stories': [91246190, 96481229]}, 'validation': {'fineweb': [0, 1162825], 'cosmopedia': [1162825, 1720043], 'textbooks': [1720043, 1912154], 'stories': [1912154, 2021309]}}, 'trainTokens': {'bytes': 192962458, 'sha256': 'cd7272ba7d77e50b384e604b6c2381228890387efd601f05a4864671131d60e7', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/cd7272ba7d77e50b384e604b6c2381228890387efd601f05a4864671131d60e7/train-bpe-u16.bin.part000.bin', 'bytes': 67108864, 'sha256': '1f2852d8fd82c1dd01a41c28e2b59254d68e1acc638d25b5a0b4d0f5de7d7a72'}, {'path': 'experiments/nebula-v28-longrun/objects/cd7272ba7d77e50b384e604b6c2381228890387efd601f05a4864671131d60e7/train-bpe-u16.bin.part001.bin', 'bytes': 67108864, 'sha256': '991637533362460ab704d6dc324d15b0f51068b295d653be5723ab71d970e790'}, {'path': 'experiments/nebula-v28-longrun/objects/cd7272ba7d77e50b384e604b6c2381228890387efd601f05a4864671131d60e7/train-bpe-u16.bin.part002.bin', 'bytes': 58744730, 'sha256': '90fe37fc2795e79386fda991087271efa018d48c4cd8fd271aefed7f278608a8'}]}, 'validationTokens': {'bytes': 4042618, 'sha256': '8aa973e396bea18da72fe26f28ab5101296fc7d4f2087241d5a654429a762f1d', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/8aa973e396bea18da72fe26f28ab5101296fc7d4f2087241d5a654429a762f1d/validation-bpe-u16.bin.part000.bin', 'bytes': 4042618, 'sha256': '8aa973e396bea18da72fe26f28ab5101296fc7d4f2087241d5a654429a762f1d'}]}, 'dataReady': True, 'sourceCard_fineweb': {'bytes': 26354, 'sha256': 'a0cc8998a20499432b28b6575f3046b714938eb8e11b8d59a1d25ddf3716061e', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/a0cc8998a20499432b28b6575f3046b714938eb8e11b8d59a1d25ddf3716061e/fineweb-source-README.md.part000.bin', 'bytes': 26354, 'sha256': 'a0cc8998a20499432b28b6575f3046b714938eb8e11b8d59a1d25ddf3716061e'}]}, 'sourceCard_cosmopedia': {'bytes': 7048, 'sha256': 'e09544d7132303ae6bbc6e156e984db0305d81561615476be49eabd4be4b8bed', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/e09544d7132303ae6bbc6e156e984db0305d81561615476be49eabd4be4b8bed/cosmopedia-source-README.md.part000.bin', 'bytes': 7048, 'sha256': 'e09544d7132303ae6bbc6e156e984db0305d81561615476be49eabd4be4b8bed'}]}, 'sourceCard_textbooks': {'bytes': 17299, 'sha256': '73c533fd5f9471d1a4bd24192b026b34d4e799e19bc2240324b0043faa023f49', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/73c533fd5f9471d1a4bd24192b026b34d4e799e19bc2240324b0043faa023f49/textbooks-source-README.md.part000.bin', 'bytes': 17299, 'sha256': '73c533fd5f9471d1a4bd24192b026b34d4e799e19bc2240324b0043faa023f49'}]}, 'sourceCard_stories': {'bytes': 1061, 'sha256': 'f67b71c3ffe761795b9428222444bfc1ec447d45cb13b00821b7e6c1ceecf7c5', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/f67b71c3ffe761795b9428222444bfc1ec447d45cb13b00821b7e6c1ceecf7c5/stories-source-README.md.part000.bin', 'bytes': 1061, 'sha256': 'f67b71c3ffe761795b9428222444bfc1ec447d45cb13b00821b7e6c1ceecf7c5'}]}, 'dataReport': {'bytes': 7879466, 'sha256': '70ec8947567153034bf72d52242e0808ce95f4681f7f60373671a23a72454728', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/70ec8947567153034bf72d52242e0808ce95f4681f7f60373671a23a72454728/data-report.json.part000.bin', 'bytes': 7879466, 'sha256': '70ec8947567153034bf72d52242e0808ce95f4681f7f60373671a23a72454728'}]}, 'foundationBaseline': {'bytes': 5059, 'sha256': 'c2adab63565a34d41482810a6982b72725da149ff1de27da98b47000c7104bac', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/c2adab63565a34d41482810a6982b72725da149ff1de27da98b47000c7104bac/foundation-baseline.json.part000.bin', 'bytes': 5059, 'sha256': 'c2adab63565a34d41482810a6982b72725da149ff1de27da98b47000c7104bac'}]}, 'foundationWorking': {'bytes': 1190857789, 'sha256': '2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part000.bin', 'bytes': 67108864, 'sha256': '0b858cac154491301f60695b1912ce4bbb7a4feee31441082a406643e6745835'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part001.bin', 'bytes': 67108864, 'sha256': '2077c20043ebfbbc72423ddc128aa6f662fc9d7d6d3653186c10e93147825264'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part002.bin', 'bytes': 67108864, 'sha256': '89fd2ce20f50b4830184d18a8c6195a4459efdf684ca132c9627dedda2685aed'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part003.bin', 'bytes': 67108864, 'sha256': '28d67eb44de26071a452f7edd25c61654862ddff93c404127c848e2f62370d75'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part004.bin', 'bytes': 67108864, 'sha256': 'bfa617567cadba972b1dee35d9c37b532184def1055c50af0281a086daa59a26'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part005.bin', 'bytes': 67108864, 'sha256': 'fb708f744516dff0956b67cbd8c253875b1b561bce875402321f2bbe384eabaa'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part006.bin', 'bytes': 67108864, 'sha256': 'c7bb9c88a9e9177f1076d489a8c311e766c5ee99c40de58ebfdad9c7ae2b87e8'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part007.bin', 'bytes': 67108864, 'sha256': '0b8ee0051b82fd8620418326b5562146bab753f30b4d1d7c1355030d0a92743d'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part008.bin', 'bytes': 67108864, 'sha256': '621c37925e99c36b7a9d9aa7b743666b3aed1c99edce488176e2fc7ca9550827'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part009.bin', 'bytes': 67108864, 'sha256': 'd4c55c25e420edfe56629b6c760cbce1e3a327d031cc74d237acbc32c4171e50'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part010.bin', 'bytes': 67108864, 'sha256': 'fe1c99a2fe8e847672a9eb70e3a32b22af43bec58b41b3d88c60cb599ae66442'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part011.bin', 'bytes': 67108864, 'sha256': 'b64958984d513f8d2463f2218a735e80ca105ed1a1acbd5fc490705f666bfb03'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part012.bin', 'bytes': 67108864, 'sha256': '50d73b52bb303dc4693a2b39e2710daa009a3917cfef247452dec33589a6093b'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part013.bin', 'bytes': 67108864, 'sha256': '5cd1a16663b564c6f75fec6acf5272c79c36ce57b956d4bc45e23830ad70ce6e'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part014.bin', 'bytes': 67108864, 'sha256': 'e3df9e9f2cee76ec14395a0e7553298422f10e62736140b7b3b3e6864f62b6dc'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part015.bin', 'bytes': 67108864, 'sha256': 'bd2a8baac0445c65168698c7e379d9a68940088938246c22a29d30f9c59c17a3'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part016.bin', 'bytes': 67108864, 'sha256': 'e6ef3b704942c75fc9936ffc7e7f0b725df370f46942c0b1fab4e0abd0d6ce78'}, {'path': 'experiments/nebula-v28-longrun/objects/2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad/foundation-working.pt.part017.bin', 'bytes': 50007101, 'sha256': '0704641be45b030fce488efab3b10984c1250d9166831995b63a277494a5688d'}]}, 'foundationLatestEvaluation': {'bytes': 5060, 'sha256': 'c0abb25dbc539eb76dd6e4b14fea200434c07aaf337414ea2e1e7afd576403cb', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/c0abb25dbc539eb76dd6e4b14fea200434c07aaf337414ea2e1e7afd576403cb/foundation-evaluation.json.part000.bin', 'bytes': 5060, 'sha256': 'c0abb25dbc539eb76dd6e4b14fea200434c07aaf337414ea2e1e7afd576403cb'}]}, 'foundationComplete': True, 'foundationFinal': {'bytes': 5507, 'sha256': '46871412e6fee796e7e17987f9328f94972c3a9c1d4613a24402a884cd61d84e', 'parts': [{'path': 'experiments/nebula-v28-longrun/objects/46871412e6fee796e7e17987f9328f94972c3a9c1d4613a24402a884cd61d84e/foundation-final.json.part000.bin', 'bytes': 5507, 'sha256': '46871412e6fee796e7e17987f9328f94972c3a9c1d4613a24402a884cd61d84e'}]}}, 'checkpointKey': 'foundationWorking', 'encoding': 'bpe'}, {'label': '250M V30 foundation 7000', 'repo': 'iiidevv/nebula-raw-training', 'revision': '6e9739ee3c5700c1b404a52931df329982f43b4a', 'state': {'format': 'nebula-v30-subword-1', 'signature': 'ce26108ca58a830fb4792566c51c4c10199d640b6bf734a697ee104f5d94d68f', 'config': {'repo': 'iiidevv/nebula-raw-training', 'rawRun': 'nebula-v18-raw-1gb', 'rawRevision': 'e167fff3bb27c36c25460fcd4c61915a5b7016ed', 'publicRun': 'nebula-v23-longrun', 'publicRevision': 'c21aafa220d33b9787791fcaf7aebf413bc62cf8', 'parentRun': 'nebula-v28-longrun', 'parentRevision': '9cfc225eae2fefe1fee5e01d77690487172d6dce', 'expectedParentUpdates': 30000, 'cleanRun': 'nebula-v29-clean-foundation', 'cleanRevision': '6eec7e5076a6e1c29d8055395179c397287094f8', 'trainBytes': 384000000, 'validationBytes': 8000000, 'tokenizerBytes': 16000000, 'sourceMix': {'fineweb': 55, 'cosmopedia': 30, 'textbooks': 10, 'stories': 5}, 'vocab': 8192, 'width': 768, 'heads': 12, 'layers': 34, 'context': 1024, 'seed': 270221, 'foundationUpdates': 10000, 'conversationUpdates': 1000, 'batch': 8, 'microbatch': 1, 'foundationLR': 3e-05, 'conversationLR': 3e-05, 'warmup': 100, 'saveEvery': 100, 'evaluateEvery': 1000}, 'steps': {'foundation': 7000}, 'targets': {'foundation': 57344000}, 'initializedFrom': 'V28 weights expanded in depth with initially identity residual layers; cleaned V29 corpus', 'qualityApproved': False, 'publicWebsiteUpdated': False, 'parentReference': {'repo': 'iiidevv/nebula-raw-training', 'run': 'nebula-v28-longrun', 'revision': '9cfc225eae2fefe1fee5e01d77690487172d6dce', 'updates': 30000, 'supervisedSubwordTokens': 245760000, 'weightSha256': '2ff87d7cf3973ba5727f7d95bf05c75a5df003986a8a922eaf732c4fedd0e4ad', 'tokenizerSha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b'}, 'tokenizerSampleBytes': {'fineweb': 8816863, 'cosmopedia': 4802434, 'textbooks': 1600631, 'stories': 800666}, 'tokenizer': {'bytes': 233872, 'sha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b/tokenizer.json.part000.bin', 'bytes': 233872, 'sha256': '67dac9cb0537aa48f58c18ab21e265456c120f3bd9ef6eb6d8a7fe894be4083b'}]}, 'cleanDataReference': {'run': 'nebula-v29-clean-foundation', 'revision': '6eec7e5076a6e1c29d8055395179c397287094f8'}, 'tokenRanges': {'validation': {'fineweb': [0, 1159423], 'cosmopedia': [1159423, 1716486], 'textbooks': [1716486, 1906898], 'stories': [1906898, 2010886]}, 'train': {'fineweb': [0, 55084443], 'cosmopedia': [55084443, 81711982], 'textbooks': [81711982, 90897183], 'stories': [90897183, 95934244]}}, 'dataReport': {'bytes': 16034794, 'sha256': 'a493fb45a665c5c2aebd4f0f9e074aff6b8939442e65b98796ca7f37e4041875', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/a493fb45a665c5c2aebd4f0f9e074aff6b8939442e65b98796ca7f37e4041875/inherited-clean-data-report.json.part000.bin', 'bytes': 16034794, 'sha256': 'a493fb45a665c5c2aebd4f0f9e074aff6b8939442e65b98796ca7f37e4041875'}]}, 'trainTokens': {'bytes': 191868488, 'sha256': 'c71b8723a7c7a0691e2e6a14a97f83200bedfacecac294779be70fe33b529b59', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/c71b8723a7c7a0691e2e6a14a97f83200bedfacecac294779be70fe33b529b59/trainTokens.bin.part000.bin', 'bytes': 67108864, 'sha256': 'bf9049e090c533cea0b2a3fece8f33a29c4b0ba68dfc3c9a55acfab70acdb7b1'}, {'path': 'experiments/nebula-v30-250m/objects/c71b8723a7c7a0691e2e6a14a97f83200bedfacecac294779be70fe33b529b59/trainTokens.bin.part001.bin', 'bytes': 67108864, 'sha256': 'b9cea5bed73808b26ee17ef202f093ebe1d01c5c3af26a85403fdfc8bd51c687'}, {'path': 'experiments/nebula-v30-250m/objects/c71b8723a7c7a0691e2e6a14a97f83200bedfacecac294779be70fe33b529b59/trainTokens.bin.part002.bin', 'bytes': 57650760, 'sha256': 'ef3cda75da2437675c8280ace37f848079c21d4a212e44e52cd6789662a87444'}]}, 'validationTokens': {'bytes': 4021772, 'sha256': 'b5e5042d7ba5eb38ccf4c1677c2479a48669c415cd39c0a77675ec547a8a4b9b', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/b5e5042d7ba5eb38ccf4c1677c2479a48669c415cd39c0a77675ec547a8a4b9b/validationTokens.bin.part000.bin', 'bytes': 4021772, 'sha256': 'b5e5042d7ba5eb38ccf4c1677c2479a48669c415cd39c0a77675ec547a8a4b9b'}]}, 'dataReady': True, 'expansion': {'originalLayers': 13, 'expandedLayers': 34, 'inheritedWeights': True, 'newResidualOutputsInitiallyZero': True, 'parentOptimizerDiscarded': True, 'parameters': 248067072}, 'attentionBackend': 'default', 'deviceAudit': {'bytes': 532, 'sha256': '523d15203b91a09ce3bf057803c215970a92aa92f8d7bb5d42446391e7478191', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/523d15203b91a09ce3bf057803c215970a92aa92f8d7bb5d42446391e7478191/device-audit.json.part000.bin', 'bytes': 532, 'sha256': '523d15203b91a09ce3bf057803c215970a92aa92f8d7bb5d42446391e7478191'}]}, 'foundationBaseline': {'bytes': 5011, 'sha256': 'b14e252245e59a99a7affa296b58368bc172f85c3390bef43e6248421a9d07d9', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/b14e252245e59a99a7affa296b58368bc172f85c3390bef43e6248421a9d07d9/foundation-baseline.json.part000.bin', 'bytes': 5011, 'sha256': 'b14e252245e59a99a7affa296b58368bc172f85c3390bef43e6248421a9d07d9'}]}, 'foundationWorking': {'bytes': 2977303078, 'sha256': '23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part000.bin', 'bytes': 67108864, 'sha256': 'e1306c943a4680104ff20924ab1676182f714c397f231ad7eed8a0d40201770c'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part001.bin', 'bytes': 67108864, 'sha256': 'fd136f8bb4ebf50b28ea7032529e7495fc286de74def6733dd5206136537ad7c'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part002.bin', 'bytes': 67108864, 'sha256': '6dfe1f1b4c51098c8f9d67b286a8a1a4066ce17862d12f641f2ac1782c558317'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part003.bin', 'bytes': 67108864, 'sha256': '3153108b6a03f5797d96fdf42de7d2f2db141257f61aaa8fd46aa824b4a7b982'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part004.bin', 'bytes': 67108864, 'sha256': '9a6df46e9717a149e34e95199934019ed485db8acac1c1f729042f20ce519772'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part005.bin', 'bytes': 67108864, 'sha256': 'ee484659c71583e904105a055d6542a8a2cb014055874a7ec394a8a2d1597609'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part006.bin', 'bytes': 67108864, 'sha256': 'bc20ad8908ab5316ba0cca40f93b4f2da0a226108a5eae430e0329db2d8f9fbd'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part007.bin', 'bytes': 67108864, 'sha256': '7310aadbe66e7767ecaea4f9530567c1f48d6e85087ad86e21e86cd89a67c8ca'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part008.bin', 'bytes': 67108864, 'sha256': 'd46be6abb5a69124de4a4ccb57ec7a6a39a94489cab2e6a887ba1b7f63bc891d'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part009.bin', 'bytes': 67108864, 'sha256': '583f2abd32366a06c3876b1db2795aec67a7c135004e6db96a483505dda9a004'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part010.bin', 'bytes': 67108864, 'sha256': '2b063d009004c0fc2c7e5cc1ad30aac4a40aec6419af269960ba5c753e9bd25f'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part011.bin', 'bytes': 67108864, 'sha256': 'ce802bd2c32ae009e762137d466522cd87888c865060393325202cc373783204'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part012.bin', 'bytes': 67108864, 'sha256': '94aacd22428c6cf1142c3b2dac0ab3d5e14a4a07bb21f7a1b7e54fba14b024a9'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part013.bin', 'bytes': 67108864, 'sha256': '035ec61667a7c754b1a913aa618185f2237c24c3928d8b59dbb307708cb1461b'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part014.bin', 'bytes': 67108864, 'sha256': 'd7135fecf14b58974aed18908243aa695bc24013f9939e6f866458db60b0f5b3'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part015.bin', 'bytes': 67108864, 'sha256': '5361994275325e3f156fb346a7bea6006f4a70b89cef3bf5ce9776ac4f75ff3a'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part016.bin', 'bytes': 67108864, 'sha256': '265cfe7e3e8e5c7cb2324cbd63f622b77ddf1863a7e1d8438f3e6b4ea29c3c36'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part017.bin', 'bytes': 67108864, 'sha256': '7fc2e9d7d297fd1943c2195ee3c4ed4f14e9f86dc65efa617097183255ca1e11'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part018.bin', 'bytes': 67108864, 'sha256': '0488b47d923b65e46832a0cc4ffd84c6188aa9db2b7c433c35bf2cb3916f391a'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part019.bin', 'bytes': 67108864, 'sha256': 'a43c2b280754eb8d7e1cbe7fc515b173d2ad7d5297c5bb2ae7fa91ab93ab1954'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part020.bin', 'bytes': 67108864, 'sha256': 'f9c1df8fd50a06044ade6bb064203887d2d61e14f0e811734edc04780228d574'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part021.bin', 'bytes': 67108864, 'sha256': '50cdc620dd1795dd463096bba43b9c2404c2b394483938ec1b6b4ec355f3de55'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part022.bin', 'bytes': 67108864, 'sha256': 'ba8ecddcea2e272b6a71a313e78c0487dfe7f6b666a6d59c318b7d1980979933'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part023.bin', 'bytes': 67108864, 'sha256': 'c22f655a394e7ab9b7aaf9694bc5271df31a9769cce9dab4776ad90d80183b16'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part024.bin', 'bytes': 67108864, 'sha256': '6b5cf0996218542a3702d420d1b9cdf09732ad7b9c686fb4a52dc2da8ede99d0'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part025.bin', 'bytes': 67108864, 'sha256': '5ab31b1682ad0e1a6f54130b91894d0c45f8a4cc59bd28403bbe0befc3985e26'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part026.bin', 'bytes': 67108864, 'sha256': '99f792d47828974fab93b222eb42ff2e2aeef26ea39464a90cce71fbe57bbfae'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part027.bin', 'bytes': 67108864, 'sha256': '5ab23687fc49fff1a09392f84440767cee5691a985f63294058c72fbc7a9978a'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part028.bin', 'bytes': 67108864, 'sha256': 'ec794b02df0339bdcc4d45ca0912cc6fa4ad4f6b939007c81501480c8551e51b'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part029.bin', 'bytes': 67108864, 'sha256': '212d29c879e0fa9f8f13ba44032a7425ef1cee6eb279191dc30f5bb4acfb7468'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part030.bin', 'bytes': 67108864, 'sha256': '0293d0181db0f879a96dd7bd687f17219e33b65ae78be0a19459647a9af2a153'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part031.bin', 'bytes': 67108864, 'sha256': '60e3ea61921d6c247a2b85f1196fc8f07b3d5730e299d7e7cdb34a8d3ad79a2b'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part032.bin', 'bytes': 67108864, 'sha256': '5e07d686e9c9b16055d6bacffb127a9b5394522744ff79d8ac002be376120ef8'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part033.bin', 'bytes': 67108864, 'sha256': '920fc4d52ebe8b87ee9ba261bc4ebcff0122177dac00e9654d853a765a7239ae'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part034.bin', 'bytes': 67108864, 'sha256': 'a5ad4ecce2bcb7a6b094d6295a5eda095bc957708c74613caef8e56ff85ea3fd'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part035.bin', 'bytes': 67108864, 'sha256': 'd266ccb1068ab7692c1557d7269c149f88aee2de1a33f2dcfa81baccbb84160c'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part036.bin', 'bytes': 67108864, 'sha256': '6e7621f46da8ea4027c005e369e16c97850e534d4dda2f4e20de2bf52a5b8df6'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part037.bin', 'bytes': 67108864, 'sha256': 'a05aaed16ac34bdb731de1fb6878cf0cb65c887b8b506ae81942102cb48ba8f5'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part038.bin', 'bytes': 67108864, 'sha256': '2787db860f78b87dce45f131726169be2beee385c3a0608c711c0815a3eae5e6'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part039.bin', 'bytes': 67108864, 'sha256': '988e744ea330a8de6dc077b696983a84f5abf5c7766b2f777a84748174d8a198'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part040.bin', 'bytes': 67108864, 'sha256': '69e3fea4cd1a540834f647be8d73d175e9ccb3edf16c4dbedcca87b8583855cd'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part041.bin', 'bytes': 67108864, 'sha256': '6d76a56453fe2a7cd5a32244f99229560140ded969a30da821e2962d91c25598'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part042.bin', 'bytes': 67108864, 'sha256': '873216dbf963d0b416df1791eb6eb3da4cffb9f72bf45a26264fecca137e440b'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part043.bin', 'bytes': 67108864, 'sha256': 'c046ff5229ff63438641ed30090238db55a172a39c2914936fd035ae242d483e'}, {'path': 'experiments/nebula-v30-250m/objects/23b01b49e96680cde9e77c9126b657a51cee301dd35e5c95518cdb69a22350a2/foundation-working.pt.part044.bin', 'bytes': 24513062, 'sha256': 'fdd2393ff6ec30c2d5c3dc776ca33f8da3f5ae59e15db1091fc04305f3321337'}]}, 'foundationLatestEvaluation': {'bytes': 5017, 'sha256': 'fc6cd73b2dd92c442f17e2a0c29c0f404bd7674fe561e4b7278ccffebd4c7cf4', 'parts': [{'path': 'experiments/nebula-v30-250m/objects/fc6cd73b2dd92c442f17e2a0c29c0f404bd7674fe561e4b7278ccffebd4c7cf4/foundation-evaluation.json.part000.bin', 'bytes': 5017, 'sha256': 'fc6cd73b2dd92c442f17e2a0c29c0f404bd7674fe561e4b7278ccffebd4c7cf4'}]}}, 'checkpointKey': 'foundationWorking', 'encoding': 'bpe'}]


In [ ]:
COMPARISON_REPORT=None
if RUN_COMPARISON:
    from compare_owned_models import compare
    COMPARISON_REPORT=compare(COMPARISON_SPECS,TOKEN,max_new_tokens=64)
    print('MODEL_COMPARISON_COMPLETED',flush=True)


In [ ]:
if PREPARE_D2:
    if not TOKEN:raise RuntimeError('Enable the existing NEBULA_HF_TOKEN Kaggle secret; never paste it into code.')
    from vd2_public_code import prepare_public
    D2_STATE=prepare_public(OUTPUT_REPO,TOKEN,PUBLIC_SOURCES,CONFIG)
    print({'d2DataReady':D2_STATE['dataReady'],'summary':D2_STATE['summary'],'trainingStarted':False},flush=True)
    if COMPARISON_REPORT is not None:
        import gzip
        from vd2_hub import Hub
        hub=Hub(OUTPUT_REPO,TOKEN);state=hub.state();hub.publish(state,{'modelComparison':gzip.compress(json.dumps(COMPARISON_REPORT).encode())})


In [ ]:
if START_D2_TRAINING:
    if not TOKEN:raise RuntimeError('Configure HF_TOKEN in Secrets.')
    TRAINER='two-t4'
    RUN_SMOKE_FIRST=True
    if TRAINER=='two-t4':
        import torch,time,socket
        from vd2_hub import Hub
        if torch.cuda.device_count()<2:raise RuntimeError('Enable two actual CUDA GPUs.')
        state=Hub(OUTPUT_REPO,TOKEN).state()
        if not state or not state.get('dataReady'):raise RuntimeError('Prepare reviewed coding sources first.')
        started=time.monotonic()
        env={**os.environ,'HF_TOKEN':TOKEN,'PYTHONPATH':os.pathsep.join([str(code_root),str(packages),os.environ.get('PYTHONPATH','')])}
        with socket.socket() as rendezvous:
            rendezvous.bind(('127.0.0.1',0));port=rendezvous.getsockname()[1]
        command=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1','--master_port='+str(port),'--nproc_per_node=2',str(code_root/'vd2_fsdp.py'),'--repo',OUTPUT_REPO,'--reserve-minutes',str(RESERVE_MINUTES)]
        if RUN_SMOKE_FIRST and not state.get('updates'):
            print('Running two-update sharded save check; private uploads may take minutes.')
            subprocess.run(command+['--hours',str(SESSION_HOURS),'--max-session-updates','2'],env=env,check=True)
        remaining=SESSION_HOURS-(time.monotonic()-started)/3600
        if remaining>RESERVE_MINUTES/60:
            print('Resuming the exact two-GPU checkpoint/optimizer for the remaining session.')
            subprocess.run(command+['--hours',str(remaining)],env=env,check=True)
        else:print('Time reserve reached. Resume the confirmed backup later.')
    elif TRAINER=='single-large-gpu':
        print(train(OUTPUT_REPO,TOKEN,SESSION_HOURS,RESERVE_MINUTES,CONFIG))
    else:raise ValueError('Unknown trainer')
